In [1]:
"""
Causal SHAP with a known (hardcoded) causal graph — mathematically rigorous γ_i.

Key design choices:
  - CAUSAL_GRAPH : edges as (parent, child) pairs — the true DAG from the translog SCM
  - γ_i computed analytically from the translog DGP using empirical E[lnX_i] (not
    ln(E[X_i])), correctly accounting for Jensen's inequality
  - State-dependence of elasticities is acknowledged; operating-point approximation
    is justified by the low coefficient of variation (CV < 3%) across the training dist.
  - PC discovery, IDA DAG sampling, and W_i regression are all removed.

Derivation of gamma_i is in gamma_derivation.md (companion file).
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
import networkx as nx

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

# Config
CONFIG = {
    'DATA_PATH':   '../../datasets/yield_preprocessed.npz',
    'MAX_PERM':    200,
    'M_RESAMPLES': 64,
    'GLOBAL_K':    30,
    'RANDOM_SEED': 42,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

FEATURE_NAMES = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

TRANSLOG_COEFFS = {
    'beta0':  2.0,
    'betaN':  0.35,  'betaI':  0.50,  'betaP':  0.25,  'betaH':  0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI': 0.04,  'betaNP': 0.02,  'betaNH': 0.01,
    'betaIP': 0.03,  'betaIH': 0.01,  'betaPH': 0.005,
}

# Causal graph
#  True structure from the DGP:
#    - All four inputs are independent roots (no inter-feature edges)
#    - Every input has a direct edge to Yield
#  Interactions (N x I, I x P, etc.) modulate edge strength but do
CAUSAL_GRAPH: List[Tuple[str, str]] = [
    ('Nitrogen',      'Yield'),
    ('Irrigation',    'Yield'),
    ('Precipitation', 'Yield'),
    ('Herbicide',     'Yield'),
]


#  GAMMA COMPUTATION  (analytically from the translog SCM)
def compute_gamma(X_train: pd.DataFrame, coeffs: dict,
                  verbose: bool = True) -> Dict[str, float]:
    """
    Derive gamma_i from the translog structural causal model.

    The local causal effect of feature X_i on ln Y is the elasticity:

        d(lnY)/d(lnX_i) = beta_i + beta_ii * lnX_i + sum_{j != i} beta_ij * lnX_j

    This elasticity is state-dependent (it varies across observations).
    We obtain a representative global causal strength by evaluating at
    the empirical mean operating point:

        E[lnX_i]   (computed from training data directly)

    Note: E[lnX] != ln(E[X]) for random variables (Jensen's inequality).
    For X ~ N(mu, sigma^2) the correction is approximately:
        E[lnX] ~= ln(mu) - sigma^2 / (2 * mu^2)
    Here we use the empirical mean of lnX directly, avoiding the approximation.

    The elasticities are then normalized so that sum(gamma_i) = 1, as required
    by the Causal SHAP weighting scheme (paper Eq. 10).

    Validity check: we compute the coefficient of variation (CV) of each
    elasticity across the training distribution. CV < 3% indicates the
    operating-point approximation introduces negligible error.
    """
    c = coeffs

    # Empirical mean of log-transformed features: E[lnX_i]
    ln_means = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    lnN = ln_means['Nitrogen']
    lnI = ln_means['Irrigation']
    lnP = ln_means['Precipitation']
    lnH = ln_means['Herbicide']

    # Analytic elasticities at the empirical mean operating point
    elasticities = {
        'Nitrogen':
            c['betaN'] + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,

        'Irrigation':
            c['betaI'] + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,

        'Precipitation':
            c['betaP'] + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,

        'Herbicide':
            c['betaH'] + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }

    for f, e in elasticities.items():
        if e <= 0:
            raise ValueError(
                f"Elasticity for {f} is non-positive ({e:.4f}) at the mean "
                "operating point. Check DGP coefficients."
            )

    # Coefficient of variation across the training distribution
    cv = {}
    for f in FEATURE_NAMES:
        e_dist = _elasticity_distribution(f, X_train, coeffs)
        cv[f] = float(np.std(e_dist) / np.mean(e_dist))

    # Normalize -> gamma_i
    total = sum(elasticities.values())
    gamma = {f: e / total for f, e in elasticities.items()}

    if verbose:
        print("\n" + "=" * 62)
        print("gamma_i derivation from translog SCM")
        print("=" * 62)
        print(f"  {'Feature':<18} {'E[lnX]':>9} {'Elasticity':>12} "
              f"{'CV(%)':>8} {'gamma_i':>9}")
        print("  " + "-" * 58)
        for f in FEATURE_NAMES:
            print(f"  {f:<18} {ln_means[f]:>9.4f} {elasticities[f]:>12.6f} "
                  f"{100*cv[f]:>7.2f}% {gamma[f]:>9.6f}")
        print("  " + "-" * 58)
        print(f"  {'Total':<18} {'':>9} {total:>12.6f} "
              f"{'':>8} {sum(gamma.values()):>9.4f}")
        print("=" * 62)
        print("  CV < 3% for all features: mean operating-point is representative.\n")

    return gamma


def _elasticity_distribution(feature: str, X: pd.DataFrame,
                              coeffs: dict) -> np.ndarray:
    """Elasticity evaluated at every training observation (quantifies state-dependence)."""
    c   = coeffs
    lnN = np.log(X['Nitrogen'].values)
    lnI = np.log(X['Irrigation'].values)
    lnP = np.log(X['Precipitation'].values)
    lnH = np.log(X['Herbicide'].values)
    return {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }[feature]


# Build dag
def build_known_dag() -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in CAUSAL_GRAPH:
        if v != 'Yield':          # feature->feature edges only in the SEM graph
            G.add_edge(u, v)
    assert nx.is_directed_acyclic_graph(G), "CAUSAL_GRAPH contains a cycle!"
    to_yield = [u for u, v in CAUSAL_GRAPH if v == 'Yield']
    print(f"Known DAG: {G.number_of_nodes()} feature nodes, "
          f"{G.number_of_edges()} inter-feature edges, "
          f"direct edges to Yield: {to_yield}")
    return G


# Linear-gaussian sem
class LinearMechanisms:
    """Linear-Gaussian SEM fitted on the known DAG for interventional sampling."""

    def __init__(self, dag: nx.DiGraph, feature_names: List[str]):
        self.dag           = dag
        self.feature_names = feature_names
        self.mechanisms: Dict[str, dict] = {}
        self.topo: List[str] = []

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        self.topo = list(nx.topological_sort(self.dag))
        for var in self.feature_names:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechanisms[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mechanisms[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention(self, S: Set[str], x_obs: pd.Series) -> Dict[str, float]:
        """One draw from P(X_{not S} | do(X_S = x_S)) via the fitted SEM."""
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            mech  = self.mechanisms[var]
            sigma = mech['sigma']
            if not mech['parents']:
                mu = mech['intercept']
            else:
                pvals = np.array([x_new.get(p, float(x_obs[p]))
                                  for p in mech['parents']])
                mu = mech['intercept'] + pvals @ mech['coef']
            x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
        return x_new


# Causal shap
class CausalSHAP:
    """
    Causal SHAP with a known DAG and analytically derived gamma_i.

    vc(S)  (paper Eq. 4):
        E_{P(X_{not S} | do(X_S = x_S))} [f(X)]
        estimated by Monte Carlo from the fitted linear-Gaussian SEM.

    phi_i  (paper Eq. 10):
        Shapley MC where each marginal contribution is weighted by gamma_i.

    Normalized (paper Eq. 11): sum(phi_i) = f(x) - E[f(X)].
    """

    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float],
                 X_train: pd.DataFrame, y_train: pd.Series):
        self.model    = model
        self.dag      = dag
        self.gamma    = gamma
        self.X_train  = X_train
        self.features = list(X_train.columns)
        self._Ef_cache = None

        print("\nFitting linear-Gaussian SEM on known DAG ...")
        self.sem = LinearMechanisms(dag, self.features).fit(X_train)
        print("SEM fitted. Ready.")

    def _Ef(self, cls: int) -> float:
        if self._Ef_cache is None:
            self._Ef_cache = self.model.predict_proba(
                self.X_train.values).mean(axis=0)
        return float(self._Ef_cache[cls])

    def _vc(self, S: Set[str], x: pd.Series, cls: int, M: int) -> float:
        total = 0.0
        for _ in range(M):
            x_s   = self.sem.sample_intervention(S, x)
            X_mat = pd.DataFrame([x_s], columns=self.features).values
            total += self.model.predict_proba(X_mat)[0][cls]
        return total / M

    def explain_instance(self, x: pd.Series,
                         n_perms: int = None, M: int = None) -> pd.Series:
        if n_perms is None: n_perms = CONFIG['MAX_PERM']
        if M       is None: M       = CONFIG['M_RESAMPLES']

        probs = self.model.predict_proba(
            pd.DataFrame([x], columns=self.features).values)[0]
        cls = int(np.argmax(probs))
        fx  = float(probs[cls])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, cls, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, cls, M)
                phi[self.features.index(feat)] += \
                    self.gamma[feat] * (v_curr - v_prev)   # Eq. 10
                v_prev, S = v_curr, S_next
        phi /= n_perms

        # Eq. 11 normalization
        sum_phi = float(np.sum(phi))
        if abs(sum_phi) > 1e-12:
            phi = phi * ((fx - self._Ef(cls)) / sum_phi)

        return pd.Series(phi, index=self.features, name='Causal_SHAP')

    def explain_global(self, X_test: pd.DataFrame, k: int = None) -> pd.Series:
        if k is None: k = CONFIG['GLOBAL_K']
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        print(f"\nGlobal Causal SHAP over {k} test instances ...")
        results = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0: print(f"  {t}/{k}")
            results.append(self.explain_instance(
                X_test.iloc[i],
                n_perms=max(10, CONFIG['MAX_PERM'] // 4),
            ))
        return pd.concat(results, axis=1).mean(axis=1).rename('Causal_SHAP_Global')


# Baselines
def compute_kernel_shap(model, X_train, X_test):
    try:
        import shap
        print("\nComputing Kernel SHAP (baseline) ...")
        bg  = shap.kmeans(X_train.values, min(100, len(X_train)))
        exp = shap.KernelExplainer(lambda x: model.predict_proba(x), bg)
        sv  = exp.shap_values(X_test.values, nsamples=100)
        glob = (np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
                if isinstance(sv, list) else np.abs(sv).mean(axis=0))
        return pd.Series(glob, index=X_test.columns, name='Kernel_SHAP')
    except Exception as e:
        print(f"Kernel SHAP skipped: {e}"); return None


def model_importance(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names,
                         name='Model_Importance')
    return None


# Plots
def plot_causal_graph(dag: nx.DiGraph, gamma: Dict[str, float],
                      elasticities: Dict[str, float]):
    G_full = dag.copy()
    G_full.add_node('Yield')
    for (u, v) in CAUSAL_GRAPH:
        if v == 'Yield':
            G_full.add_edge(u, 'Yield')

    fig, ax = plt.subplots(figsize=(9, 5))
    pos = {
        'Nitrogen':      (0.12, 0.80),
        'Irrigation':    (0.38, 0.80),
        'Precipitation': (0.64, 0.80),
        'Herbicide':     (0.88, 0.80),
        'Yield':         (0.50, 0.15),
    }
    colors = ['#5DCAA5' if n != 'Yield' else '#7F77DD' for n in G_full.nodes()]
    nx.draw_networkx_nodes(G_full, pos, node_color=colors,
                           node_size=2200, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G_full, pos, font_size=8.5, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G_full, pos, edge_color='#555', arrows=True,
                           arrowsize=20, width=2.5,
                           connectionstyle='arc3,rad=0.05', ax=ax)
    for (u, v) in CAUSAL_GRAPH:
        if v == 'Yield' and u in gamma:
            x0, y0 = pos[u]; x1, y1 = pos['Yield']
            ax.text((x0+x1)/2 + 0.04, (y0+y1)/2,
                    f'e={elasticities[u]:.3f}\ng={gamma[u]:.3f}',
                    fontsize=7.5, color='#0C447C', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.25', fc='white',
                              alpha=0.85, ec='#B5D4F4', lw=0.8))
    ax.set_title(
        'True causal graph (translog SCM)\n'
        'e = local elasticity at E[lnX],  g = normalized causal weight (gamma_i)',
        fontsize=10, fontweight='bold'
    )
    ax.axis('off'); fig.tight_layout()
    return fig


def plot_attributions(df: pd.DataFrame):
    n_cols = df.shape[1]
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 5), sharey=False)
    if n_cols == 1: axes = [axes]
    for ax, col in zip(axes, df.columns):
        vals   = df[col].values
        colors = ['#3B8BD4' if v >= 0 else '#E24B4A' for v in vals]
        bars   = ax.barh(df.index.tolist(), vals, color=colors,
                         edgecolor='white', height=0.55)
        ax.axvline(0, color='#888', linewidth=0.8, linestyle='--')
        for bar, val in zip(bars, vals):
            ax.text(val + (0.0005 if val >= 0 else -0.0005),
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center',
                    ha='left' if val >= 0 else 'right', fontsize=9)
        ax.set_title(col, fontweight='bold')
        ax.set_xlabel('Attribution score')
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(
        'Feature Attribution — Yield (known causal graph, analytic gamma_i)',
        fontsize=11, fontweight='bold', y=1.01
    )
    fig.tight_layout()
    return fig


# Main
def main():
    print("\n" + "="*70)
    print("CAUSAL SHAP — Known Causal Graph (analytic gamma_i from translog SCM)")
    print("="*70 + "\n")

    try:
        data = np.load(CONFIG['DATA_PATH'], allow_pickle=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' not found."); return

    X_train = pd.DataFrame(data['X_train'], columns=list(data['feature_names']))
    X_test  = pd.DataFrame(data['X_test'],  columns=list(data['feature_names']))
    y_train = pd.Series(data['y_train'].astype(int))
    y_test  = pd.Series(data['y_test'].astype(int))
    print(f"Loaded: X_train={X_train.shape}, X_test={X_test.shape}")

    # gamma_i is derived from unscaled X_train (log-space elasticities from DGP)
    gamma = compute_gamma(X_train, TRANSLOG_COEFFS, verbose=True)

    # Raw elasticities (for plot annotation only)
    lnm = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    c   = TRANSLOG_COEFFS
    elasticities = {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnm['Nitrogen']  + c['betaNI']*lnm['Irrigation']
            + c['betaNP']*lnm['Precipitation'] + c['betaNH']*lnm['Herbicide'],
        'Irrigation':
            c['betaI']  + c['betaII']*lnm['Irrigation'] + c['betaNI']*lnm['Nitrogen']
            + c['betaIP']*lnm['Precipitation'] + c['betaIH']*lnm['Herbicide'],
        'Precipitation':
            c['betaP']  + c['betaPP']*lnm['Precipitation'] + c['betaNP']*lnm['Nitrogen']
            + c['betaIP']*lnm['Irrigation']   + c['betaPH']*lnm['Herbicide'],
        'Herbicide':
            c['betaH']  + c['betaHH']*lnm['Herbicide']  + c['betaNH']*lnm['Nitrogen']
            + c['betaIH']*lnm['Irrigation']   + c['betaPH']*lnm['Precipitation'],
    }

    # Scale for model training / SEM fitting
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    Xte = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

    n_classes = int(len(np.unique(y_train)))
    print(f"Training model ({n_classes} classes) ...")
    if HAVE_XGB:
        model = XGBClassifier(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
            objective='multi:softprob', num_class=n_classes,
        )
    else:
        model = RandomForestClassifier(
            n_estimators=160, max_depth=6,
            random_state=CONFIG['RANDOM_SEED'])
    model.fit(Xtr.values, y_train.values)
    y_pred = model.predict(Xte.values)
    print(f"Accuracy={accuracy_score(y_test, y_pred):.4f}  "
          f"MacroF1={f1_score(y_test, y_pred, average='macro'):.4f}")

    dag   = build_known_dag()
    fig_g = plot_causal_graph(dag, gamma, elasticities)
    fig_g.savefig('causal_graph_yield.png', dpi=150, bbox_inches='tight')
    print("Saved causal_graph_yield.png")

    cshap      = CausalSHAP(model=model, dag=dag, gamma=gamma,
                             X_train=Xtr, y_train=y_train)
    global_phi = cshap.explain_global(Xte, k=CONFIG['GLOBAL_K'])
    print("\nGlobal Causal SHAP (sorted by |attribution|):")
    print(global_phi.sort_values(key=abs, ascending=False).to_string())

    kshap = compute_kernel_shap(model, Xtr, Xte)
    mimp  = model_importance(model, list(Xtr.columns))

    cols = {'Causal_SHAP': global_phi}
    if kshap is not None: cols['Kernel_SHAP']     = kshap
    if mimp  is not None: cols['Model_Importance'] = mimp
    df = pd.DataFrame(cols)
    df.to_csv('attribution_scores_yield.csv')
    print("\n" + df.round(6).to_string())
    print("\nSaved attribution_scores_yield.csv")

    fig_a = plot_attributions(df)
    fig_a.savefig('attribution_comparison_yield.png', dpi=150, bbox_inches='tight')
    print("Saved attribution_comparison_yield.png")


if __name__ == '__main__':
    main()


CAUSAL SHAP — Known Causal Graph (analytic gamma_i from translog SCM)

Loaded: X_train=(800, 4), X_test=(200, 4)

gamma_i derivation from translog SCM
  Feature               E[lnX]   Elasticity    CV(%)   gamma_i
  ----------------------------------------------------------
  Nitrogen              5.4763     0.660202    1.79%  0.311820
  Irrigation            5.7561     0.789182    1.47%  0.372739
  Precipitation         5.3331     0.520288    1.64%  0.245738
  Herbicide             8.2822     0.147578    2.15%  0.069703
  ----------------------------------------------------------
  Total                            2.117250             1.0000
  CV < 3% for all features: mean operating-point is representative.

Training model (3 classes) ...
Accuracy=0.8050  MacroF1=0.8081
Known DAG: 4 feature nodes, 0 inter-feature edges, direct edges to Yield: ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']
Saved causal_graph_yield.png

Fitting linear-Gaussian SEM on known DAG ...
SEM fitted

Kernel SHAP skipped: Data must be 1-dimensional, got ndarray of shape (4, 3) instead

               Causal_SHAP  Model_Importance
Nitrogen          0.066144          0.181695
Irrigation        0.346474          0.450784
Precipitation     0.103026          0.274794
Herbicide         0.001521          0.092727

Saved attribution_scores_yield.csv
Saved attribution_comparison_yield.png


In [2]:
"""
Causal SHAP with a known (hardcoded) causal graph — mathematically rigorous γ_i.

Key design choices:
  - CAUSAL_GRAPH : edges as (parent, child) pairs — the true DAG from the translog SCM
  - γ_i computed analytically from the translog DGP using empirical E[lnX_i] (not
    ln(E[X_i])), correctly accounting for Jensen's inequality
  - State-dependence of elasticities is acknowledged; operating-point approximation
    is justified by the low coefficient of variation (CV < 3%) across the training dist.
  - PC discovery, IDA DAG sampling, and W_i regression are all removed.

Derivation of gamma_i is in gamma_derivation.md (companion file).
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
import networkx as nx

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

# Config
CONFIG = {
    'DATA_PATH':   '../../datasets/yield_preprocessed.npz',
    'MAX_PERM':    500,
    'M_RESAMPLES': 128,
    'GLOBAL_K':    200,
    'RANDOM_SEED': 42,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

FEATURE_NAMES = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

TRANSLOG_COEFFS = {
    'beta0':  2.0,
    'betaN':  0.35,  'betaI':  0.50,  'betaP':  0.25,  'betaH':  0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI': 0.04,  'betaNP': 0.02,  'betaNH': 0.01,
    'betaIP': 0.03,  'betaIH': 0.01,  'betaPH': 0.005,
}

# Causal graph
#  True structure from the DGP:
#    - All four inputs are independent roots (no inter-feature edges)
#    - Every input has a direct edge to Yield
#  Interactions (N x I, I x P, etc.) modulate edge strength but do
CAUSAL_GRAPH: List[Tuple[str, str]] = [
    ('Nitrogen',      'Yield'),
    ('Irrigation',    'Yield'),
    ('Precipitation', 'Yield'),
    ('Herbicide',     'Yield'),
]


#  GAMMA COMPUTATION  (analytically from the translog SCM)
def compute_gamma(X_train: pd.DataFrame, coeffs: dict,
                  verbose: bool = True) -> Dict[str, float]:
    """
    Derive gamma_i from the translog structural causal model.

    The local causal effect of feature X_i on ln Y is the elasticity:

        d(lnY)/d(lnX_i) = beta_i + beta_ii * lnX_i + sum_{j != i} beta_ij * lnX_j

    This elasticity is state-dependent (it varies across observations).
    We obtain a representative global causal strength by evaluating at
    the empirical mean operating point:

        E[lnX_i]   (computed from training data directly)

    Note: E[lnX] != ln(E[X]) for random variables (Jensen's inequality).
    For X ~ N(mu, sigma^2) the correction is approximately:
        E[lnX] ~= ln(mu) - sigma^2 / (2 * mu^2)
    Here we use the empirical mean of lnX directly, avoiding the approximation.

    The elasticities are then normalized so that sum(gamma_i) = 1, as required
    by the Causal SHAP weighting scheme (paper Eq. 10).

    Validity check: we compute the coefficient of variation (CV) of each
    elasticity across the training distribution. CV < 3% indicates the
    operating-point approximation introduces negligible error.
    """
    c = coeffs

    # Empirical mean of log-transformed features: E[lnX_i]
    ln_means = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    lnN = ln_means['Nitrogen']
    lnI = ln_means['Irrigation']
    lnP = ln_means['Precipitation']
    lnH = ln_means['Herbicide']

    # Analytic elasticities at the empirical mean operating point
    elasticities = {
        'Nitrogen':
            c['betaN'] + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,

        'Irrigation':
            c['betaI'] + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,

        'Precipitation':
            c['betaP'] + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,

        'Herbicide':
            c['betaH'] + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }

    for f, e in elasticities.items():
        if e <= 0:
            raise ValueError(
                f"Elasticity for {f} is non-positive ({e:.4f}) at the mean "
                "operating point. Check DGP coefficients."
            )

    # Coefficient of variation across the training distribution
    cv = {}
    for f in FEATURE_NAMES:
        e_dist = _elasticity_distribution(f, X_train, coeffs)
        cv[f] = float(np.std(e_dist) / np.mean(e_dist))

    # Normalize -> gamma_i
    total = sum(elasticities.values())
    gamma = {f: e / total for f, e in elasticities.items()}

    if verbose:
        print("\n" + "=" * 62)
        print("gamma_i derivation from translog SCM")
        print("=" * 62)
        print(f"  {'Feature':<18} {'E[lnX]':>9} {'Elasticity':>12} "
              f"{'CV(%)':>8} {'gamma_i':>9}")
        print("  " + "-" * 58)
        for f in FEATURE_NAMES:
            print(f"  {f:<18} {ln_means[f]:>9.4f} {elasticities[f]:>12.6f} "
                  f"{100*cv[f]:>7.2f}% {gamma[f]:>9.6f}")
        print("  " + "-" * 58)
        print(f"  {'Total':<18} {'':>9} {total:>12.6f} "
              f"{'':>8} {sum(gamma.values()):>9.4f}")
        print("=" * 62)
        print("  CV < 3% for all features: mean operating-point is representative.\n")

    return gamma


def _elasticity_distribution(feature: str, X: pd.DataFrame,
                              coeffs: dict) -> np.ndarray:
    """Elasticity evaluated at every training observation (quantifies state-dependence)."""
    c   = coeffs
    lnN = np.log(X['Nitrogen'].values)
    lnI = np.log(X['Irrigation'].values)
    lnP = np.log(X['Precipitation'].values)
    lnH = np.log(X['Herbicide'].values)
    return {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }[feature]


# Build dag
def build_known_dag() -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in CAUSAL_GRAPH:
        if v != 'Yield':          # feature->feature edges only in the SEM graph
            G.add_edge(u, v)
    assert nx.is_directed_acyclic_graph(G), "CAUSAL_GRAPH contains a cycle!"
    to_yield = [u for u, v in CAUSAL_GRAPH if v == 'Yield']
    print(f"Known DAG: {G.number_of_nodes()} feature nodes, "
          f"{G.number_of_edges()} inter-feature edges, "
          f"direct edges to Yield: {to_yield}")
    return G


# Linear-gaussian sem
class LinearMechanisms:
    """Linear-Gaussian SEM fitted on the known DAG for interventional sampling."""

    def __init__(self, dag: nx.DiGraph, feature_names: List[str]):
        self.dag           = dag
        self.feature_names = feature_names
        self.mechanisms: Dict[str, dict] = {}
        self.topo: List[str] = []

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        self.topo = list(nx.topological_sort(self.dag))
        for var in self.feature_names:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechanisms[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mechanisms[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention(self, S: Set[str], x_obs: pd.Series) -> Dict[str, float]:
        """One draw from P(X_{not S} | do(X_S = x_S)) via the fitted SEM."""
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            mech  = self.mechanisms[var]
            sigma = mech['sigma']
            if not mech['parents']:
                mu = mech['intercept']
            else:
                pvals = np.array([x_new.get(p, float(x_obs[p]))
                                  for p in mech['parents']])
                mu = mech['intercept'] + pvals @ mech['coef']
            x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
        return x_new


# Causal shap
class CausalSHAP:
    """
    Causal SHAP with a known DAG and analytically derived gamma_i.

    vc(S)  (paper Eq. 4):
        E_{P(X_{not S} | do(X_S = x_S))} [f(X)]
        estimated by Monte Carlo from the fitted linear-Gaussian SEM.

    phi_i  (paper Eq. 10):
        Shapley MC where each marginal contribution is weighted by gamma_i.

    Normalized (paper Eq. 11): sum(phi_i) = f(x) - E[f(X)].
    """

    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float],
                 X_train: pd.DataFrame, y_train: pd.Series):
        self.model    = model
        self.dag      = dag
        self.gamma    = gamma
        self.X_train  = X_train
        self.features = list(X_train.columns)
        self._Ef_cache = None

        print("\nFitting linear-Gaussian SEM on known DAG ...")
        self.sem = LinearMechanisms(dag, self.features).fit(X_train)
        print("SEM fitted. Ready.")

    def _Ef(self, cls: int) -> float:
        if self._Ef_cache is None:
            self._Ef_cache = self.model.predict_proba(
                self.X_train.values).mean(axis=0)
        return float(self._Ef_cache[cls])

    def _vc(self, S: Set[str], x: pd.Series, cls: int, M: int) -> float:
        total = 0.0
        for _ in range(M):
            x_s   = self.sem.sample_intervention(S, x)
            X_mat = pd.DataFrame([x_s], columns=self.features).values
            total += self.model.predict_proba(X_mat)[0][cls]
        return total / M

    def explain_instance(self, x: pd.Series,
                         n_perms: int = None, M: int = None) -> pd.Series:
        if n_perms is None: n_perms = CONFIG['MAX_PERM']
        if M       is None: M       = CONFIG['M_RESAMPLES']

        probs = self.model.predict_proba(
            pd.DataFrame([x], columns=self.features).values)[0]
        cls = int(np.argmax(probs))
        fx  = float(probs[cls])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, cls, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, cls, M)
                phi[self.features.index(feat)] += \
                    self.gamma[feat] * (v_curr - v_prev)   # Eq. 10
                v_prev, S = v_curr, S_next
        phi /= n_perms

        # Eq. 11 normalization
        sum_phi = float(np.sum(phi))
        if abs(sum_phi) > 1e-12:
            phi = phi * ((fx - self._Ef(cls)) / sum_phi)

        return pd.Series(phi, index=self.features, name='Causal_SHAP')

    def explain_global(self, X_test: pd.DataFrame, k: int = None) -> pd.Series:
        if k is None: k = CONFIG['GLOBAL_K']
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        print(f"\nGlobal Causal SHAP over {k} test instances ...")
        results = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0: print(f"  {t}/{k}")
            results.append(self.explain_instance(
                X_test.iloc[i],
                n_perms=max(10, CONFIG['MAX_PERM'] // 4),
            ))
        return pd.concat(results, axis=1).mean(axis=1).rename('Causal_SHAP_Global')


# Baselines
def compute_kernel_shap(model, X_train, X_test):
    try:
        import shap
        print("\nComputing Kernel SHAP (baseline) ...")
        bg  = shap.kmeans(X_train.values, min(100, len(X_train)))
        exp = shap.KernelExplainer(lambda x: model.predict_proba(x), bg)
        sv  = exp.shap_values(X_test.values, nsamples=100)
        glob = (np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
                if isinstance(sv, list) else np.abs(sv).mean(axis=0))
        return pd.Series(glob, index=X_test.columns, name='Kernel_SHAP')
    except Exception as e:
        print(f"Kernel SHAP skipped: {e}"); return None


def model_importance(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names,
                         name='Model_Importance')
    return None


# Plots
def plot_causal_graph(dag: nx.DiGraph, gamma: Dict[str, float],
                      elasticities: Dict[str, float]):
    G_full = dag.copy()
    G_full.add_node('Yield')
    for (u, v) in CAUSAL_GRAPH:
        if v == 'Yield':
            G_full.add_edge(u, 'Yield')

    fig, ax = plt.subplots(figsize=(9, 5))
    pos = {
        'Nitrogen':      (0.12, 0.80),
        'Irrigation':    (0.38, 0.80),
        'Precipitation': (0.64, 0.80),
        'Herbicide':     (0.88, 0.80),
        'Yield':         (0.50, 0.15),
    }
    colors = ['#5DCAA5' if n != 'Yield' else '#7F77DD' for n in G_full.nodes()]
    nx.draw_networkx_nodes(G_full, pos, node_color=colors,
                           node_size=2200, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G_full, pos, font_size=8.5, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G_full, pos, edge_color='#555', arrows=True,
                           arrowsize=20, width=2.5,
                           connectionstyle='arc3,rad=0.05', ax=ax)
    for (u, v) in CAUSAL_GRAPH:
        if v == 'Yield' and u in gamma:
            x0, y0 = pos[u]; x1, y1 = pos['Yield']
            ax.text((x0+x1)/2 + 0.04, (y0+y1)/2,
                    f'e={elasticities[u]:.3f}\ng={gamma[u]:.3f}',
                    fontsize=7.5, color='#0C447C', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.25', fc='white',
                              alpha=0.85, ec='#B5D4F4', lw=0.8))
    ax.set_title(
        'True causal graph (translog SCM)\n'
        'e = local elasticity at E[lnX],  g = normalized causal weight (gamma_i)',
        fontsize=10, fontweight='bold'
    )
    ax.axis('off'); fig.tight_layout()
    return fig


def plot_attributions(df: pd.DataFrame):
    n_cols = df.shape[1]
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 5), sharey=False)
    if n_cols == 1: axes = [axes]
    for ax, col in zip(axes, df.columns):
        vals   = df[col].values
        colors = ['#3B8BD4' if v >= 0 else '#E24B4A' for v in vals]
        bars   = ax.barh(df.index.tolist(), vals, color=colors,
                         edgecolor='white', height=0.55)
        ax.axvline(0, color='#888', linewidth=0.8, linestyle='--')
        for bar, val in zip(bars, vals):
            ax.text(val + (0.0005 if val >= 0 else -0.0005),
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center',
                    ha='left' if val >= 0 else 'right', fontsize=9)
        ax.set_title(col, fontweight='bold')
        ax.set_xlabel('Attribution score')
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(
        'Feature Attribution — Yield (known causal graph, analytic gamma_i)',
        fontsize=11, fontweight='bold', y=1.01
    )
    fig.tight_layout()
    return fig


# Main
def main():
    print("\n" + "="*70)
    print("CAUSAL SHAP — Known Causal Graph (analytic gamma_i from translog SCM)")
    print("="*70 + "\n")

    try:
        data = np.load(CONFIG['DATA_PATH'], allow_pickle=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' not found."); return

    X_train = pd.DataFrame(data['X_train'], columns=list(data['feature_names']))
    X_test  = pd.DataFrame(data['X_test'],  columns=list(data['feature_names']))
    y_train = pd.Series(data['y_train'].astype(int))
    y_test  = pd.Series(data['y_test'].astype(int))
    print(f"Loaded: X_train={X_train.shape}, X_test={X_test.shape}")

    # gamma_i is derived from unscaled X_train (log-space elasticities from DGP)
    gamma = compute_gamma(X_train, TRANSLOG_COEFFS, verbose=True)

    # Raw elasticities (for plot annotation only)
    lnm = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    c   = TRANSLOG_COEFFS
    elasticities = {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnm['Nitrogen']  + c['betaNI']*lnm['Irrigation']
            + c['betaNP']*lnm['Precipitation'] + c['betaNH']*lnm['Herbicide'],
        'Irrigation':
            c['betaI']  + c['betaII']*lnm['Irrigation'] + c['betaNI']*lnm['Nitrogen']
            + c['betaIP']*lnm['Precipitation'] + c['betaIH']*lnm['Herbicide'],
        'Precipitation':
            c['betaP']  + c['betaPP']*lnm['Precipitation'] + c['betaNP']*lnm['Nitrogen']
            + c['betaIP']*lnm['Irrigation']   + c['betaPH']*lnm['Herbicide'],
        'Herbicide':
            c['betaH']  + c['betaHH']*lnm['Herbicide']  + c['betaNH']*lnm['Nitrogen']
            + c['betaIH']*lnm['Irrigation']   + c['betaPH']*lnm['Precipitation'],
    }

    # Scale for model training / SEM fitting
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    Xte = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

    n_classes = int(len(np.unique(y_train)))
    print(f"Training model ({n_classes} classes) ...")
    if HAVE_XGB:
        model = XGBClassifier(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
            objective='multi:softprob', num_class=n_classes,
        )
    else:
        model = RandomForestClassifier(
            n_estimators=160, max_depth=6,
            random_state=CONFIG['RANDOM_SEED'])
    model.fit(Xtr.values, y_train.values)
    y_pred = model.predict(Xte.values)
    print(f"Accuracy={accuracy_score(y_test, y_pred):.4f}  "
          f"MacroF1={f1_score(y_test, y_pred, average='macro'):.4f}")

    dag   = build_known_dag()
    fig_g = plot_causal_graph(dag, gamma, elasticities)
    fig_g.savefig('causal_graph_yield.png', dpi=150, bbox_inches='tight')
    print("Saved causal_graph_yield.png")

    cshap      = CausalSHAP(model=model, dag=dag, gamma=gamma,
                             X_train=Xtr, y_train=y_train)
    global_phi = cshap.explain_global(Xte, k=CONFIG['GLOBAL_K'])
    print("\nGlobal Causal SHAP (sorted by |attribution|):")
    print(global_phi.sort_values(key=abs, ascending=False).to_string())

    kshap = compute_kernel_shap(model, Xtr, Xte)
    mimp  = model_importance(model, list(Xtr.columns))

    cols = {'Causal_SHAP': global_phi}
    if kshap is not None: cols['Kernel_SHAP']     = kshap
    if mimp  is not None: cols['Model_Importance'] = mimp
    df = pd.DataFrame(cols)
    df.to_csv('attribution_scores_yield.csv')
    print("\n" + df.round(6).to_string())
    print("\nSaved attribution_scores_yield.csv")

    fig_a = plot_attributions(df)
    fig_a.savefig('attribution_comparison_yield2.png', dpi=150, bbox_inches='tight')
    print("Saved attribution_comparison_yield.png")


if __name__ == '__main__':
    main()


CAUSAL SHAP — Known Causal Graph (analytic gamma_i from translog SCM)

Loaded: X_train=(800, 4), X_test=(200, 4)

gamma_i derivation from translog SCM
  Feature               E[lnX]   Elasticity    CV(%)   gamma_i
  ----------------------------------------------------------
  Nitrogen              5.4763     0.660202    1.79%  0.311820
  Irrigation            5.7561     0.789182    1.47%  0.372739
  Precipitation         5.3331     0.520288    1.64%  0.245738
  Herbicide             8.2822     0.147578    2.15%  0.069703
  ----------------------------------------------------------
  Total                            2.117250             1.0000
  CV < 3% for all features: mean operating-point is representative.

Training model (3 classes) ...
Accuracy=0.8050  MacroF1=0.8081
Known DAG: 4 feature nodes, 0 inter-feature edges, direct edges to Yield: ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']
Saved causal_graph_yield.png

Fitting linear-Gaussian SEM on known DAG ...
SEM fitted

Kernel SHAP skipped: Data must be 1-dimensional, got ndarray of shape (4, 3) instead

               Causal_SHAP  Model_Importance
Nitrogen          0.072310          0.181695
Irrigation        0.327322          0.450784
Precipitation     0.105983          0.274794
Herbicide         0.004000          0.092727

Saved attribution_scores_yield.csv
Saved attribution_comparison_yield.png


In [3]:
"""
Causal SHAP with a known (hardcoded) causal graph — mathematically rigorous γ_i.

Key design choices:
  - CAUSAL_GRAPH : edges as (parent, child) pairs — the true DAG from the translog SCM
  - γ_i computed analytically from the translog DGP using empirical E[lnX_i] (not
    ln(E[X_i])), correctly accounting for Jensen's inequality
  - State-dependence of elasticities is acknowledged; operating-point approximation
    is justified by the low coefficient of variation (CV < 3%) across the training dist.
  - PC discovery, IDA DAG sampling, and W_i regression are all removed.

Derivation of gamma_i is in gamma_derivation.md (companion file).
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
import networkx as nx

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

HAVE_XGB=False

# Config
CONFIG = {
    'DATA_PATH':   '../../datasets/yield_preprocessed.npz',
    'MAX_PERM':    200,
    'M_RESAMPLES': 64,
    'GLOBAL_K':    30,
    'RANDOM_SEED': 42,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

FEATURE_NAMES = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

TRANSLOG_COEFFS = {
    'beta0':  2.0,
    'betaN':  0.35,  'betaI':  0.50,  'betaP':  0.25,  'betaH':  0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI': 0.04,  'betaNP': 0.02,  'betaNH': 0.01,
    'betaIP': 0.03,  'betaIH': 0.01,  'betaPH': 0.005,
}

# Causal graph
#  True structure from the DGP:
#    - All four inputs are independent roots (no inter-feature edges)
#    - Every input has a direct edge to Yield
#  Interactions (N x I, I x P, etc.) modulate edge strength but do
CAUSAL_GRAPH: List[Tuple[str, str]] = [
    ('Nitrogen',      'Yield'),
    ('Irrigation',    'Yield'),
    ('Precipitation', 'Yield'),
    ('Herbicide',     'Yield'),
]


#  GAMMA COMPUTATION  (analytically from the translog SCM)
def compute_gamma(X_train: pd.DataFrame, coeffs: dict,
                  verbose: bool = True) -> Dict[str, float]:
    """
    Derive gamma_i from the translog structural causal model.

    The local causal effect of feature X_i on ln Y is the elasticity:

        d(lnY)/d(lnX_i) = beta_i + beta_ii * lnX_i + sum_{j != i} beta_ij * lnX_j

    This elasticity is state-dependent (it varies across observations).
    We obtain a representative global causal strength by evaluating at
    the empirical mean operating point:

        E[lnX_i]   (computed from training data directly)

    Note: E[lnX] != ln(E[X]) for random variables (Jensen's inequality).
    For X ~ N(mu, sigma^2) the correction is approximately:
        E[lnX] ~= ln(mu) - sigma^2 / (2 * mu^2)
    Here we use the empirical mean of lnX directly, avoiding the approximation.

    The elasticities are then normalized so that sum(gamma_i) = 1, as required
    by the Causal SHAP weighting scheme (paper Eq. 10).

    Validity check: we compute the coefficient of variation (CV) of each
    elasticity across the training distribution. CV < 3% indicates the
    operating-point approximation introduces negligible error.
    """
    c = coeffs

    # Empirical mean of log-transformed features: E[lnX_i]
    ln_means = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    lnN = ln_means['Nitrogen']
    lnI = ln_means['Irrigation']
    lnP = ln_means['Precipitation']
    lnH = ln_means['Herbicide']

    # Analytic elasticities at the empirical mean operating point
    elasticities = {
        'Nitrogen':
            c['betaN'] + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,

        'Irrigation':
            c['betaI'] + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,

        'Precipitation':
            c['betaP'] + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,

        'Herbicide':
            c['betaH'] + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }

    for f, e in elasticities.items():
        if e <= 0:
            raise ValueError(
                f"Elasticity for {f} is non-positive ({e:.4f}) at the mean "
                "operating point. Check DGP coefficients."
            )

    # Coefficient of variation across the training distribution
    cv = {}
    for f in FEATURE_NAMES:
        e_dist = _elasticity_distribution(f, X_train, coeffs)
        cv[f] = float(np.std(e_dist) / np.mean(e_dist))

    # Normalize -> gamma_i
    total = sum(elasticities.values())
    gamma = {f: e / total for f, e in elasticities.items()}

    if verbose:
        print("\n" + "=" * 62)
        print("gamma_i derivation from translog SCM")
        print("=" * 62)
        print(f"  {'Feature':<18} {'E[lnX]':>9} {'Elasticity':>12} "
              f"{'CV(%)':>8} {'gamma_i':>9}")
        print("  " + "-" * 58)
        for f in FEATURE_NAMES:
            print(f"  {f:<18} {ln_means[f]:>9.4f} {elasticities[f]:>12.6f} "
                  f"{100*cv[f]:>7.2f}% {gamma[f]:>9.6f}")
        print("  " + "-" * 58)
        print(f"  {'Total':<18} {'':>9} {total:>12.6f} "
              f"{'':>8} {sum(gamma.values()):>9.4f}")
        print("=" * 62)
        print("  CV < 3% for all features: mean operating-point is representative.\n")

    return gamma


def _elasticity_distribution(feature: str, X: pd.DataFrame,
                              coeffs: dict) -> np.ndarray:
    """Elasticity evaluated at every training observation (quantifies state-dependence)."""
    c   = coeffs
    lnN = np.log(X['Nitrogen'].values)
    lnI = np.log(X['Irrigation'].values)
    lnP = np.log(X['Precipitation'].values)
    lnH = np.log(X['Herbicide'].values)
    return {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }[feature]


# Build dag
def build_known_dag() -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in CAUSAL_GRAPH:
        if v != 'Yield':          # feature->feature edges only in the SEM graph
            G.add_edge(u, v)
    assert nx.is_directed_acyclic_graph(G), "CAUSAL_GRAPH contains a cycle!"
    to_yield = [u for u, v in CAUSAL_GRAPH if v == 'Yield']
    print(f"Known DAG: {G.number_of_nodes()} feature nodes, "
          f"{G.number_of_edges()} inter-feature edges, "
          f"direct edges to Yield: {to_yield}")
    return G


# Linear-gaussian sem
class LinearMechanisms:
    """Linear-Gaussian SEM fitted on the known DAG for interventional sampling."""

    def __init__(self, dag: nx.DiGraph, feature_names: List[str]):
        self.dag           = dag
        self.feature_names = feature_names
        self.mechanisms: Dict[str, dict] = {}
        self.topo: List[str] = []

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        self.topo = list(nx.topological_sort(self.dag))
        for var in self.feature_names:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechanisms[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mechanisms[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention(self, S: Set[str], x_obs: pd.Series) -> Dict[str, float]:
        """One draw from P(X_{not S} | do(X_S = x_S)) via the fitted SEM."""
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            mech  = self.mechanisms[var]
            sigma = mech['sigma']
            if not mech['parents']:
                mu = mech['intercept']
            else:
                pvals = np.array([x_new.get(p, float(x_obs[p]))
                                  for p in mech['parents']])
                mu = mech['intercept'] + pvals @ mech['coef']
            x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
        return x_new


# Causal shap
class CausalSHAP:
    """
    Causal SHAP with a known DAG and analytically derived gamma_i.

    vc(S)  (paper Eq. 4):
        E_{P(X_{not S} | do(X_S = x_S))} [f(X)]
        estimated by Monte Carlo from the fitted linear-Gaussian SEM.

    phi_i  (paper Eq. 10):
        Shapley MC where each marginal contribution is weighted by gamma_i.

    Normalized (paper Eq. 11): sum(phi_i) = f(x) - E[f(X)].
    """

    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float],
                 X_train: pd.DataFrame, y_train: pd.Series):
        self.model    = model
        self.dag      = dag
        self.gamma    = gamma
        self.X_train  = X_train
        self.features = list(X_train.columns)
        self._Ef_cache = None

        print("\nFitting linear-Gaussian SEM on known DAG ...")
        self.sem = LinearMechanisms(dag, self.features).fit(X_train)
        print("SEM fitted. Ready.")

    def _Ef(self, cls: int) -> float:
        if self._Ef_cache is None:
            self._Ef_cache = self.model.predict_proba(
                self.X_train.values).mean(axis=0)
        return float(self._Ef_cache[cls])

    def _vc(self, S: Set[str], x: pd.Series, cls: int, M: int) -> float:
        total = 0.0
        for _ in range(M):
            x_s   = self.sem.sample_intervention(S, x)
            X_mat = pd.DataFrame([x_s], columns=self.features).values
            total += self.model.predict_proba(X_mat)[0][cls]
        return total / M

    def explain_instance(self, x: pd.Series,
                         n_perms: int = None, M: int = None) -> pd.Series:
        if n_perms is None: n_perms = CONFIG['MAX_PERM']
        if M       is None: M       = CONFIG['M_RESAMPLES']

        probs = self.model.predict_proba(
            pd.DataFrame([x], columns=self.features).values)[0]
        cls = int(np.argmax(probs))
        fx  = float(probs[cls])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, cls, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, cls, M)
                phi[self.features.index(feat)] += \
                    self.gamma[feat] * (v_curr - v_prev)   # Eq. 10
                v_prev, S = v_curr, S_next
        phi /= n_perms

        # Eq. 11 normalization
        sum_phi = float(np.sum(phi))
        if abs(sum_phi) > 1e-12:
            phi = phi * ((fx - self._Ef(cls)) / sum_phi)

        return pd.Series(phi, index=self.features, name='Causal_SHAP')

    def explain_global(self, X_test: pd.DataFrame, k: int = None) -> pd.Series:
        if k is None: k = CONFIG['GLOBAL_K']
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        print(f"\nGlobal Causal SHAP over {k} test instances ...")
        results = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0: print(f"  {t}/{k}")
            results.append(self.explain_instance(
                X_test.iloc[i],
                n_perms=max(10, CONFIG['MAX_PERM'] // 4),
            ))
        return pd.concat(results, axis=1).mean(axis=1).rename('Causal_SHAP_Global')


# Baselines
def compute_kernel_shap(model, X_train, X_test):
    try:
        import shap
        print("\nComputing Kernel SHAP (baseline) ...")
        bg  = shap.kmeans(X_train.values, min(100, len(X_train)))
        exp = shap.KernelExplainer(lambda x: model.predict_proba(x), bg)
        sv  = exp.shap_values(X_test.values, nsamples=100)
        glob = (np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
                if isinstance(sv, list) else np.abs(sv).mean(axis=0))
        return pd.Series(glob, index=X_test.columns, name='Kernel_SHAP')
    except Exception as e:
        print(f"Kernel SHAP skipped: {e}"); return None


def model_importance(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names,
                         name='Model_Importance')
    return None


# Plots
def plot_causal_graph(dag: nx.DiGraph, gamma: Dict[str, float],
                      elasticities: Dict[str, float]):
    G_full = dag.copy()
    G_full.add_node('Yield')
    for (u, v) in CAUSAL_GRAPH:
        if v == 'Yield':
            G_full.add_edge(u, 'Yield')

    fig, ax = plt.subplots(figsize=(9, 5))
    pos = {
        'Nitrogen':      (0.12, 0.80),
        'Irrigation':    (0.38, 0.80),
        'Precipitation': (0.64, 0.80),
        'Herbicide':     (0.88, 0.80),
        'Yield':         (0.50, 0.15),
    }
    colors = ['#5DCAA5' if n != 'Yield' else '#7F77DD' for n in G_full.nodes()]
    nx.draw_networkx_nodes(G_full, pos, node_color=colors,
                           node_size=2200, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G_full, pos, font_size=8.5, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G_full, pos, edge_color='#555', arrows=True,
                           arrowsize=20, width=2.5,
                           connectionstyle='arc3,rad=0.05', ax=ax)
    for (u, v) in CAUSAL_GRAPH:
        if v == 'Yield' and u in gamma:
            x0, y0 = pos[u]; x1, y1 = pos['Yield']
            ax.text((x0+x1)/2 + 0.04, (y0+y1)/2,
                    f'e={elasticities[u]:.3f}\ng={gamma[u]:.3f}',
                    fontsize=7.5, color='#0C447C', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.25', fc='white',
                              alpha=0.85, ec='#B5D4F4', lw=0.8))
    ax.set_title(
        'True causal graph (translog SCM)\n'
        'e = local elasticity at E[lnX],  g = normalized causal weight (gamma_i)',
        fontsize=10, fontweight='bold'
    )
    ax.axis('off'); fig.tight_layout()
    return fig


def plot_attributions(df: pd.DataFrame):
    n_cols = df.shape[1]
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 5), sharey=False)
    if n_cols == 1: axes = [axes]
    for ax, col in zip(axes, df.columns):
        vals   = df[col].values
        colors = ['#3B8BD4' if v >= 0 else '#E24B4A' for v in vals]
        bars   = ax.barh(df.index.tolist(), vals, color=colors,
                         edgecolor='white', height=0.55)
        ax.axvline(0, color='#888', linewidth=0.8, linestyle='--')
        for bar, val in zip(bars, vals):
            ax.text(val + (0.0005 if val >= 0 else -0.0005),
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center',
                    ha='left' if val >= 0 else 'right', fontsize=9)
        ax.set_title(col, fontweight='bold')
        ax.set_xlabel('Attribution score')
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(
        'Feature Attribution — Yield (known causal graph, analytic gamma_i)',
        fontsize=11, fontweight='bold', y=1.01
    )
    fig.tight_layout()
    return fig


# Main
def main():
    print("\n" + "="*70)
    print("CAUSAL SHAP — Known Causal Graph (analytic gamma_i from translog SCM)")
    print("="*70 + "\n")

    try:
        data = np.load(CONFIG['DATA_PATH'], allow_pickle=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' not found."); return

    X_train = pd.DataFrame(data['X_train'], columns=list(data['feature_names']))
    X_test  = pd.DataFrame(data['X_test'],  columns=list(data['feature_names']))
    y_train = pd.Series(data['y_train'].astype(int))
    y_test  = pd.Series(data['y_test'].astype(int))
    print(f"Loaded: X_train={X_train.shape}, X_test={X_test.shape}")

    # gamma_i is derived from unscaled X_train (log-space elasticities from DGP)
    gamma = compute_gamma(X_train, TRANSLOG_COEFFS, verbose=True)

    # Raw elasticities (for plot annotation only)
    lnm = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    c   = TRANSLOG_COEFFS
    elasticities = {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnm['Nitrogen']  + c['betaNI']*lnm['Irrigation']
            + c['betaNP']*lnm['Precipitation'] + c['betaNH']*lnm['Herbicide'],
        'Irrigation':
            c['betaI']  + c['betaII']*lnm['Irrigation'] + c['betaNI']*lnm['Nitrogen']
            + c['betaIP']*lnm['Precipitation'] + c['betaIH']*lnm['Herbicide'],
        'Precipitation':
            c['betaP']  + c['betaPP']*lnm['Precipitation'] + c['betaNP']*lnm['Nitrogen']
            + c['betaIP']*lnm['Irrigation']   + c['betaPH']*lnm['Herbicide'],
        'Herbicide':
            c['betaH']  + c['betaHH']*lnm['Herbicide']  + c['betaNH']*lnm['Nitrogen']
            + c['betaIH']*lnm['Irrigation']   + c['betaPH']*lnm['Precipitation'],
    }

    # Scale for model training / SEM fitting
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    Xte = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

    n_classes = int(len(np.unique(y_train)))
    print(f"Training model ({n_classes} classes) ...")
    if HAVE_XGB:
        model = XGBClassifier(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
            objective='multi:softprob', num_class=n_classes,
        )
    else:
        model = RandomForestClassifier(
            n_estimators=160, max_depth=6,
            random_state=CONFIG['RANDOM_SEED'])
    model.fit(Xtr.values, y_train.values)
    y_pred = model.predict(Xte.values)
    print(f"Accuracy={accuracy_score(y_test, y_pred):.4f}  "
          f"MacroF1={f1_score(y_test, y_pred, average='macro'):.4f}")

    dag   = build_known_dag()
    fig_g = plot_causal_graph(dag, gamma, elasticities)
    fig_g.savefig('causal_graph_yield.png', dpi=150, bbox_inches='tight')
    print("Saved causal_graph_yield.png")

    cshap      = CausalSHAP(model=model, dag=dag, gamma=gamma,
                             X_train=Xtr, y_train=y_train)
    global_phi = cshap.explain_global(Xte, k=CONFIG['GLOBAL_K'])
    print("\nGlobal Causal SHAP (sorted by |attribution|):")
    print(global_phi.sort_values(key=abs, ascending=False).to_string())

    kshap = compute_kernel_shap(model, Xtr, Xte)
    mimp  = model_importance(model, list(Xtr.columns))

    cols = {'Causal_SHAP': global_phi}
    if kshap is not None: cols['Kernel_SHAP']     = kshap
    if mimp  is not None: cols['Model_Importance'] = mimp
    df = pd.DataFrame(cols)
    df.to_csv('attribution_scores_yield.csv')
    print("\n" + df.round(6).to_string())
    print("\nSaved attribution_scores_yield.csv")

    fig_a = plot_attributions(df)
    fig_a.savefig('attribution_comparison_yield.png', dpi=150, bbox_inches='tight')
    print("Saved attribution_comparison_yield.png")


if __name__ == '__main__':
    main()


CAUSAL SHAP — Known Causal Graph (analytic gamma_i from translog SCM)

Loaded: X_train=(800, 4), X_test=(200, 4)

gamma_i derivation from translog SCM
  Feature               E[lnX]   Elasticity    CV(%)   gamma_i
  ----------------------------------------------------------
  Nitrogen              5.4763     0.660202    1.79%  0.311820
  Irrigation            5.7561     0.789182    1.47%  0.372739
  Precipitation         5.3331     0.520288    1.64%  0.245738
  Herbicide             8.2822     0.147578    2.15%  0.069703
  ----------------------------------------------------------
  Total                            2.117250             1.0000
  CV < 3% for all features: mean operating-point is representative.

Training model (3 classes) ...
Accuracy=0.8250  MacroF1=0.8280
Known DAG: 4 feature nodes, 0 inter-feature edges, direct edges to Yield: ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']
Saved causal_graph_yield.png

Fitting linear-Gaussian SEM on known DAG ...
SEM fitted

Kernel SHAP skipped: Data must be 1-dimensional, got ndarray of shape (4, 3) instead

               Causal_SHAP  Model_Importance
Nitrogen          0.031640          0.167071
Irrigation        0.290490          0.530923
Precipitation     0.088671          0.252476
Herbicide         0.000163          0.049531

Saved attribution_scores_yield.csv
Saved attribution_comparison_yield.png


In [5]:
"""
Causal SHAP — Yield dataset, REGRESSION version.

Identical to the classification script (causal_shap_known_graph.py) except:

  Changed
  -------
  - DATA_PATH     : yield_regression_preprocessed.npz  (continuous y, float64)
  - Model         : XGBRegressor / RandomForestRegressor
  - Metrics       : RMSE, MAE, R²
  - _Ef()         : scalar E[f(X)] = mean(model.predict(X_train))
  - _vc()         : model.predict()[0]  — scalar, no class axis
  - explain_instance(): f(x) is model.predict(x)[0], scalar
  - explain_global(): returns (global_signed, global_abs) — both reported
  - Kernel SHAP   : uses model.predict  (regression; sv is 2-D, no class axis)
  - Output cols   : GroundTruth_Gamma | Causal_SHAP_Signed | Causal_SHAP_AbsMean
  - Global metrics: Pearson r, Spearman r, MAE between
                    Causal_SHAP_AbsMean (normalised) and GroundTruth_Gamma

  Unchanged
  ---------
  - CONFIG (except DATA_PATH)
  - FEATURE_NAMES, TRANSLOG_COEFFS, CAUSAL_GRAPH
  - compute_gamma(), _elasticity_distribution()  — model-agnostic
  - build_known_dag()
  - LinearMechanisms (SEM)
  - Eq. 10 weighting (gamma_i * delta_vc), Eq. 11 normalisation
  - plot_causal_graph(), plot_attributions()
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import networkx as nx

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

# Config
CONFIG = {
    'DATA_PATH':   '../../datasets/yield_regression_preprocessed.npz',
    'MAX_PERM':    200,
    'M_RESAMPLES': 64,
    'GLOBAL_K':    30,
    'RANDOM_SEED': 42,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

FEATURE_NAMES = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']

#  TRANSLOG COEFFICIENTS  (unchanged)
TRANSLOG_COEFFS = {
    'beta0':  2.0,
    'betaN':  0.35,  'betaI':  0.50,  'betaP':  0.25,  'betaH':  0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI': 0.04,  'betaNP': 0.02,  'betaNH': 0.01,
    'betaIP': 0.03,  'betaIH': 0.01,  'betaPH': 0.005,
}

#  CAUSAL GRAPH  (unchanged)
CAUSAL_GRAPH: List[Tuple[str, str]] = [
    ('Nitrogen',      'Yield'),
    ('Irrigation',    'Yield'),
    ('Precipitation', 'Yield'),
    ('Herbicide',     'Yield'),
]

#  GAMMA COMPUTATION  (unchanged - model-agnostic)
def compute_gamma(X_train: pd.DataFrame, coeffs: dict,
                  verbose: bool = True) -> Dict[str, float]:
    c = coeffs
    ln_means = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    lnN = ln_means['Nitrogen']
    lnI = ln_means['Irrigation']
    lnP = ln_means['Precipitation']
    lnH = ln_means['Herbicide']

    elasticities = {
        'Nitrogen':
            c['betaN'] + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI'] + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP'] + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH'] + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }

    for f, e in elasticities.items():
        if e <= 0:
            raise ValueError(f"Elasticity for {f} is non-positive ({e:.4f}).")

    cv = {}
    for f in FEATURE_NAMES:
        e_dist = _elasticity_distribution(f, X_train, coeffs)
        cv[f] = float(np.std(e_dist) / np.mean(e_dist))

    total = sum(elasticities.values())
    gamma = {f: e / total for f, e in elasticities.items()}

    if verbose:
        print("\n" + "=" * 62)
        print("gamma_i derivation from translog SCM  (unchanged from classification)")
        print("=" * 62)
        print(f"  {'Feature':<18} {'E[lnX]':>9} {'Elasticity':>12} "
              f"{'CV(%)':>8} {'gamma_i':>9}")
        print("  " + "-" * 58)
        for f in FEATURE_NAMES:
            print(f"  {f:<18} {ln_means[f]:>9.4f} {elasticities[f]:>12.6f} "
                  f"{100*cv[f]:>7.2f}% {gamma[f]:>9.6f}")
        print("  " + "-" * 58)
        print(f"  {'Total':<18} {'':>9} {total:>12.6f} "
              f"{'':>8} {sum(gamma.values()):>9.4f}")
        print("=" * 62)
        print("  CV < 3%: mean operating-point is representative.\n")
    return gamma


def _elasticity_distribution(feature: str, X: pd.DataFrame,
                              coeffs: dict) -> np.ndarray:
    c   = coeffs
    lnN = np.log(X['Nitrogen'].values)
    lnI = np.log(X['Irrigation'].values)
    lnP = np.log(X['Precipitation'].values)
    lnH = np.log(X['Herbicide'].values)
    return {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }[feature]

#  BUILD DAG  (unchanged)
def build_known_dag() -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in CAUSAL_GRAPH:
        if v != 'Yield':
            G.add_edge(u, v)
    assert nx.is_directed_acyclic_graph(G), "CAUSAL_GRAPH contains a cycle!"
    to_yield = [u for u, v in CAUSAL_GRAPH if v == 'Yield']
    print(f"Known DAG: {G.number_of_nodes()} feature nodes, "
          f"{G.number_of_edges()} inter-feature edges, "
          f"direct edges to Yield: {to_yield}")
    return G

#  LINEAR-GAUSSIAN SEM  (unchanged)
class LinearMechanisms:
    def __init__(self, dag: nx.DiGraph, feature_names: List[str]):
        self.dag           = dag
        self.feature_names = feature_names
        self.mechanisms: Dict[str, dict] = {}
        self.topo: List[str] = []

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        self.topo = list(nx.topological_sort(self.dag))
        for var in self.feature_names:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechanisms[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mechanisms[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention(self, S: Set[str], x_obs: pd.Series) -> Dict[str, float]:
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            mech  = self.mechanisms[var]
            sigma = mech['sigma']
            if not mech['parents']:
                mu = mech['intercept']
            else:
                pvals = np.array([x_new.get(p, float(x_obs[p]))
                                  for p in mech['parents']])
                mu = mech['intercept'] + pvals @ mech['coef']
            x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
        return x_new

# Causal shap - regression
#  Three changes vs the classification CausalSHAP:
#    1. _Ef()  -> scalar mean(model.predict(X_train))
#    2. _vc()  -> model.predict()[0], no class axis
#    3. explain_instance() -> f(x) = model.predict(x)[0]
#  Eq. 10 weighting and Eq. 11 normalisation are character-for-character
#  identical; they operate on continuous SPI values instead of probabilities.
class CausalSHAP:
    """Causal SHAP — regression variant. phi units = yield units."""

    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float],
                 X_train: pd.DataFrame, y_train: pd.Series):
        self.model    = model
        self.dag      = dag
        self.gamma    = gamma
        self.X_train  = X_train
        self.features = list(X_train.columns)
        self._Ef_val  = None          # scalar cache

        print("\nFitting linear-Gaussian SEM on known DAG ...")
        self.sem = LinearMechanisms(dag, self.features).fit(X_train)
        print("SEM fitted. Ready.")

    def _Ef(self) -> float:
        if self._Ef_val is None:
            self._Ef_val = float(self.model.predict(self.X_train.values).mean())
        return self._Ef_val

    def _vc(self, S: Set[str], x: pd.Series, M: int) -> float:
        total = 0.0
        for _ in range(M):
            x_s   = self.sem.sample_intervention(S, x)
            X_mat = pd.DataFrame([x_s], columns=self.features).values
            total += float(self.model.predict(X_mat)[0])
        return total / M

    def explain_instance(self, x: pd.Series,
                         n_perms: int = None, M: int = None) -> pd.Series:
        if n_perms is None: n_perms = CONFIG['MAX_PERM']
        if M       is None: M       = CONFIG['M_RESAMPLES']

        fx = float(self.model.predict(
            pd.DataFrame([x], columns=self.features).values)[0])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm   = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, M)
                phi[self.features.index(feat)] += \
                    self.gamma[feat] * (v_curr - v_prev)    # Eq. 10
                v_prev, S = v_curr, S_next
        phi /= n_perms

        sum_phi = float(np.sum(phi))
        if abs(sum_phi) > 1e-12:
            phi = phi * ((fx - self._Ef()) / sum_phi)      # Eq. 11

        return pd.Series(phi, index=self.features, name='Causal_SHAP')

    # ── returns signed + abs (regression analogue of classification version) ─
    def explain_global(self, X_test: pd.DataFrame, k: int = None):
        """
        Returns:
          global_signed  — mean(phi_i)    direction + cancellation
          global_abs     — mean(|phi_i|)  magnitude / importance
        """
        if k is None: k = CONFIG['GLOBAL_K']
        k   = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        print(f"\nGlobal Causal SHAP over {k} test instances ...")
        results = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0: print(f"  {t}/{k}")
            results.append(self.explain_instance(
                X_test.iloc[i],
                n_perms=max(10, CONFIG['MAX_PERM'] // 4),
            ))
        phi_mat       = pd.concat(results, axis=1)
        global_signed = phi_mat.mean(axis=1).rename('Causal_SHAP_Signed')
        global_abs    = phi_mat.abs().mean(axis=1).rename('Causal_SHAP_AbsMean')
        return global_signed, global_abs

# Global ranking metrics
def compute_global_metrics(gamma: Dict[str, float],
                           global_abs: pd.Series) -> None:
    gt_vec   = np.array([gamma[f]             for f in FEATURE_NAMES])
    shap_raw = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
    shap_norm = shap_raw / shap_raw.sum() if shap_raw.sum() > 1e-12 else shap_raw

    mae            = float(np.mean(np.abs(shap_norm - gt_vec)))
    pearson_r, pp  = pearsonr(shap_norm, gt_vec)
    spearman_r, sp = spearmanr(shap_norm, gt_vec)

    print("\n--- Global ranking metrics "
          "(Causal_SHAP_AbsMean normalised vs GroundTruth_Gamma) ---")
    print(f"  MAE        = {mae:.6f}")
    print(f"  Pearson r  = {pearson_r:.4f}  (p={pp:.4f})")
    print(f"  Spearman r = {spearman_r:.4f}  (p={sp:.4f})")

#  BASELINES  (kernel SHAP uses model.predict - regression)
def compute_kernel_shap(model, X_train, X_test):
    try:
        import shap
        print("\nComputing Kernel SHAP (baseline) ...")
        bg  = shap.kmeans(X_train.values, min(100, len(X_train)))
        exp = shap.KernelExplainer(model.predict, bg)   # predict, not predict_proba
        sv  = exp.shap_values(X_test.values, nsamples=100)
        # Regression: sv is (n_samples x n_features) - no class axis
        glob = np.abs(sv).mean(axis=0)
        return pd.Series(glob, index=X_test.columns, name='Kernel_SHAP')
    except Exception as e:
        print(f"Kernel SHAP skipped: {e}"); return None


def model_importance(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names,
                         name='Model_Importance')
    return None

#  PLOTS  (unchanged from classification version)
def plot_causal_graph(dag: nx.DiGraph, gamma: Dict[str, float],
                      elasticities: Dict[str, float]):
    G_full = dag.copy()
    G_full.add_node('Yield')
    for (u, v) in CAUSAL_GRAPH:
        if v == 'Yield':
            G_full.add_edge(u, 'Yield')

    fig, ax = plt.subplots(figsize=(9, 5))
    pos = {
        'Nitrogen':      (0.12, 0.80),
        'Irrigation':    (0.38, 0.80),
        'Precipitation': (0.64, 0.80),
        'Herbicide':     (0.88, 0.80),
        'Yield':         (0.50, 0.15),
    }
    colors = ['#5DCAA5' if n != 'Yield' else '#7F77DD' for n in G_full.nodes()]
    nx.draw_networkx_nodes(G_full, pos, node_color=colors,
                           node_size=2200, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G_full, pos, font_size=8.5, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G_full, pos, edge_color='#555', arrows=True,
                           arrowsize=20, width=2.5,
                           connectionstyle='arc3,rad=0.05', ax=ax)
    for (u, v) in CAUSAL_GRAPH:
        if v == 'Yield' and u in gamma:
            x0, y0 = pos[u]; x1, y1 = pos['Yield']
            ax.text((x0+x1)/2 + 0.04, (y0+y1)/2,
                    f'e={elasticities[u]:.3f}\ng={gamma[u]:.3f}',
                    fontsize=7.5, color='#0C447C', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.25', fc='white',
                              alpha=0.85, ec='#B5D4F4', lw=0.8))
    ax.set_title(
        'True causal graph (translog SCM) — regression\n'
        'e = local elasticity at E[lnX],  g = normalized causal weight (gamma_i)',
        fontsize=10, fontweight='bold'
    )
    ax.axis('off'); fig.tight_layout()
    return fig


def plot_attributions(df: pd.DataFrame):
    n_cols = df.shape[1]
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 5), sharey=False)
    if n_cols == 1: axes = [axes]
    for ax, col in zip(axes, df.columns):
        vals   = df[col].values
        colors = ['#3B8BD4' if v >= 0 else '#E24B4A' for v in vals]
        bars   = ax.barh(df.index.tolist(), vals, color=colors,
                         edgecolor='white', height=0.55)
        ax.axvline(0, color='#888', linewidth=0.8, linestyle='--')
        for bar, val in zip(bars, vals):
            ax.text(val + (0.0005 if val >= 0 else -0.0005),
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center',
                    ha='left' if val >= 0 else 'right', fontsize=9)
        ax.set_title(col, fontweight='bold')
        ax.set_xlabel('Attribution score')
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(
        'Feature Attribution — Yield Regression\n'
        '(known causal graph, analytic gamma_i)',
        fontsize=11, fontweight='bold', y=1.01
    )
    fig.tight_layout()
    return fig

# Main
def main():
    print("\n" + "="*70)
    print("CAUSAL SHAP — Yield REGRESSION (continuous yield target)")
    print("="*70 + "\n")

    # ── Load NPZ (continuous y already stored as float64) ─────────────────
    try:
        data = np.load(CONFIG['DATA_PATH'], allow_pickle=True)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' not found."); return

    X_train = pd.DataFrame(data['X_train'], columns=list(data['feature_names']))
    X_test  = pd.DataFrame(data['X_test'],  columns=list(data['feature_names']))
    y_train = data['y_train'].astype(float)   # continuous, not int
    y_test  = data['y_test'].astype(float)
    print(f"Loaded: X_train={X_train.shape}, X_test={X_test.shape}")
    print(f"y_train: min={y_train.min():.1f}  max={y_train.max():.1f}  "
          f"mean={y_train.mean():.1f}  std={y_train.std():.1f}\n")

    # ── gamma_i (unchanged - derived from unscaled X_train) ───────────────
    gamma = compute_gamma(X_train, TRANSLOG_COEFFS, verbose=True)

    # ── Raw elasticities (for plot annotation only) ───────────────────────
    lnm = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    c   = TRANSLOG_COEFFS
    elasticities = {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnm['Nitrogen']  + c['betaNI']*lnm['Irrigation']
            + c['betaNP']*lnm['Precipitation'] + c['betaNH']*lnm['Herbicide'],
        'Irrigation':
            c['betaI']  + c['betaII']*lnm['Irrigation'] + c['betaNI']*lnm['Nitrogen']
            + c['betaIP']*lnm['Precipitation'] + c['betaIH']*lnm['Herbicide'],
        'Precipitation':
            c['betaP']  + c['betaPP']*lnm['Precipitation'] + c['betaNP']*lnm['Nitrogen']
            + c['betaIP']*lnm['Irrigation']   + c['betaPH']*lnm['Herbicide'],
        'Herbicide':
            c['betaH']  + c['betaHH']*lnm['Herbicide']  + c['betaNH']*lnm['Nitrogen']
            + c['betaIH']*lnm['Irrigation']   + c['betaPH']*lnm['Precipitation'],
    }

    # ── Scale ─────────────────────────────────────────────────────────────
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    Xte = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

    # ── Train regression model ────────────────────────────────────────────
    print("Training regression model ...")
    if HAVE_XGB:
        model = XGBRegressor(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
        )
        print("Using XGBRegressor")
    else:
        model = RandomForestRegressor(
            n_estimators=160, max_depth=6,
            random_state=CONFIG['RANDOM_SEED'])
        print("Using RandomForestRegressor")

    model.fit(Xtr.values, y_train)
    y_pred = model.predict(Xte.values)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae  = float(mean_absolute_error(y_test, y_pred))
    r2   = float(r2_score(y_test, y_pred))
    print(f"RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}\n")

    # ── Build DAG and save graph ──────────────────────────────────────────
    dag   = build_known_dag()
    fig_g = plot_causal_graph(dag, gamma, elasticities)
    fig_g.savefig('causal_graph_yield_regression.png', dpi=150, bbox_inches='tight')
    print("Saved causal_graph_yield_regression.png")

    # ── Causal SHAP ───────────────────────────────────────────────────────
    cshap = CausalSHAP(model=model, dag=dag, gamma=gamma,
                       X_train=Xtr, y_train=pd.Series(y_train))
    global_signed, global_abs = cshap.explain_global(Xte, k=CONFIG['GLOBAL_K'])

    print("\nGlobal Causal SHAP — signed mean (direction + cancellation):")
    print(global_signed.sort_values(key=abs, ascending=False).to_string())
    print("\nGlobal Causal SHAP — mean absolute attribution (importance):")
    print(global_abs.sort_values(ascending=False).to_string())

    # ── Global ranking metrics ────────────────────────────────────────────
    compute_global_metrics(gamma, global_abs)

    # ── Baselines ─────────────────────────────────────────────────────────
    kshap = compute_kernel_shap(model, Xtr, Xte)
    mimp  = model_importance(model, FEATURE_NAMES)

    # ── Output table - GT gamma first column ─────────────────────────────
    gt_gamma = pd.Series(gamma, name='GroundTruth_Gamma')
    cols = {
        'GroundTruth_Gamma':   gt_gamma,
        'Causal_SHAP_Signed':  global_signed,
        'Causal_SHAP_AbsMean': global_abs,
    }
    if kshap is not None: cols['Kernel_SHAP']     = kshap
    if mimp  is not None: cols['Model_Importance'] = mimp
    df = pd.DataFrame(cols)
    df.to_csv('attribution_scores_yield_regression.csv')
    print("\n" + df.round(6).to_string())
    print("\nSaved attribution_scores_yield_regression.csv")

    fig_a = plot_attributions(df)
    fig_a.savefig('attribution_comparison_yield_regression.png',
                  dpi=150, bbox_inches='tight')
    print("Saved attribution_comparison_yield_regression.png")


if __name__ == '__main__':
    main()


CAUSAL SHAP — Yield REGRESSION (continuous yield target)

Loaded: X_train=(800, 4), X_test=(200, 4)
y_train: min=27188.2  max=151221.2  mean=87032.9  std=20717.0


gamma_i derivation from translog SCM  (unchanged from classification)
  Feature               E[lnX]   Elasticity    CV(%)   gamma_i
  ----------------------------------------------------------
  Nitrogen              5.4763     0.660202    1.79%  0.311820
  Irrigation            5.7561     0.789182    1.47%  0.372739
  Precipitation         5.3331     0.520288    1.64%  0.245738
  Herbicide             8.2822     0.147578    2.15%  0.069703
  ----------------------------------------------------------
  Total                            2.117250             1.0000
  CV < 3%: mean operating-point is representative.

Training regression model ...
Using XGBRegressor
RMSE=5351.2529  MAE=4155.1981  R²=0.9336

Known DAG: 4 feature nodes, 0 inter-feature edges, direct edges to Yield: ['Nitrogen', 'Irrigation', 'Precipitation', 'Her


               GroundTruth_Gamma  Causal_SHAP_Signed  Causal_SHAP_AbsMean   Kernel_SHAP  Model_Importance
Nitrogen                0.311820         -768.376117          4988.884038   5004.647867          0.117369
Irrigation              0.372739        -3134.194509         14638.406475  12682.924658          0.620348
Precipitation           0.245738           98.626637          5429.907380   6752.047130          0.232211
Herbicide               0.069703           66.338390           237.326271   1222.316211          0.030072

Saved attribution_scores_yield_regression.csv
Saved attribution_comparison_yield_regression.png


In [1]:
"""
causal_shap_yield_robustness.py
================================
Robustness analysis of Causal SHAP attributions under SEM edge-weight
perturbation for the Yield REGRESSION dataset.

PRECISE CLAIM TESTED
---------------------
  "Causal SHAP attributions for yield prediction are sensitive to errors
   in SEM edge-weight estimation, even when the KNOWN causal graph structure
   (all features -> Yield, no inter-feature edges) is held fixed and correct.
   This isolates SEM-parameter sensitivity from graph-structure uncertainty."

WHAT THIS DOES NOT TEST
------------------------
  Missing edges, wrong edge directions, or extra edges.

KEY DIFFERENCES FROM SOIL ROBUSTNESS SCRIPT
--------------------------------------------
  - Data loaded from .npz (not CSV); X already split into X_train / X_test.
  - Gamma derived from the translog SCM elasticities (not ground_truth.csv).
  - Known DAG has NO inter-feature edges (all 4 features are root nodes).
    => SEM coefficients only control the marginal distributions (intercept +
       sigma) of each feature — there are no structural beta_ij to perturb.
    => ARM 1 perturbation therefore targets the MARGINAL DISTRIBUTION
       parameters (intercept/mean and sigma) via additive noise, which is
       the correct sensitivity test for a DAG with no inter-feature edges.
  - ARM 1 perturbation for marginal nodes:
        mu'  = mu  * (1 + epsilon * delta_mu),   delta_mu  ~ U(-1,1)
        sig' = sig * (1 + epsilon * |delta_sig|), delta_sig ~ U(-1,1)
       (sigma must stay positive, hence absolute value on delta_sig)
  - ARM 2 (estimation noise) refits marginal statistics on subsamples;
    shows how training-set size affects attribution stability.

TWO EXPERIMENTAL ARMS
---------------------
  ARM 1 — Adversarial / worst-case perturbation
    Baseline SEM fitted on full X_train.
    For each epsilon in EPSILONS:
      Perturb marginal parameters (mu, sigma) multiplicatively.
      Repeat N_TRIALS times (different delta draws, same instances).
      Measure: MAD(phi_eps, phi_0) and Spearman rho of feature rankings.
    Find eps* = first epsilon where noise-corrected MAD > 5% of |phi_0|.

  ARM 2 — Estimation noise (realistic / constructive)
    SEM marginal statistics refit on subsets of X_train of size n.
    Measure: MAD(phi_n, phi_full) as function of n.
    Model, graph, gamma, and test instances are FROZEN.

KEY VALIDITY CONTROLS
---------------------
  1. SHARED RANDOM SEEDS  — instance_indices, perm_seeds, noise_seeds
     generated ONCE, reused for every run. Only SEM parameters change.
  2. epsilon=0 NOISE FLOOR  — pure MC estimator noise; all reported MAD
     is noise-corrected: MAD_corr = max(0, MAD(eps) - MAD(0)).
  3. MONOTONICITY CHECK  — corrected MAD must increase with epsilon.
  4. GAMMA FROM TRANSLOG SCM  — gamma_i is fixed from analytic elasticities,
     independent of the SEM fitting.

OUTPUTS
-------
  arm1_adversarial_yield.png
  arm2_estimation_yield.png
  gamma_vs_sensitivity_yield.png
  arm1_scores_yield.csv
  arm2_scores_yield.csv
"""

import copy
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx

from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from typing import Dict, List, Optional, Set, Tuple

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"figure.dpi": 120})

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

# Configuration

CFG = {
    "data_path":     "../../datasets/yield_regression_preprocessed.npz",

    # Experiment settings
    "random_seed":   42,

    # Causal SHAP MC settings
    "n_perms":       30,      # permutation samples per instance
    "M_per_sem":     16,      # interventional MC draws per coalition
    "k_instances":   5,       # test instances for global SHAP

    # ARM 1 - adversarial sweep
    "epsilons":      [0.0, 0.05, 0.10, 0.30, 0.50, 1.0],
    "n_trials":      3,       # perturbation draws per epsilon
    "breakdown_pct": 0.05,    # eps* threshold = 5% of mean |phi_0|

    # ARM 2 - estimation noise
    "subsample_sizes": [50, 100, 250, 500, 750],
}


FEATURE_NAMES = ["Nitrogen", "Irrigation", "Precipitation", "Herbicide"]

CAUSAL_GRAPH: List[Tuple[str, str]] = [
    ("Nitrogen",      "Yield"),
    ("Irrigation",    "Yield"),
    ("Precipitation", "Yield"),
    ("Herbicide",     "Yield"),
]

TRANSLOG_COEFFS = {
    "beta0":  2.0,
    "betaN":  0.35, "betaI":  0.50, "betaP":  0.25, "betaH":  0.05,
    "betaNN": -0.02, "betaII": -0.03, "betaPP": -0.01, "betaHH": -0.005,
    "betaNI": 0.04,  "betaNP": 0.02,  "betaNH": 0.01,
    "betaIP": 0.03,  "betaIH": 0.01,  "betaPH": 0.005,
}

WEAK_PARAM_THRESHOLD = 1e-3   # params below this are not perturbed


# 1. data loading

def load_yield_data(cfg: dict):
    """Load and scale the yield dataset from NPZ."""
    data    = np.load(cfg["data_path"], allow_pickle=True)
    X_train = pd.DataFrame(data["X_train"], columns=list(data["feature_names"]))
    X_test  = pd.DataFrame(data["X_test"],  columns=list(data["feature_names"]))
    y_train = data["y_train"].astype(float)
    y_test  = data["y_test"].astype(float)

    print(f"Loaded: X_train={X_train.shape}  X_test={X_test.shape}")
    print(f"y_train: min={y_train.min():.2f}  max={y_train.max():.2f}  "
          f"mean={y_train.mean():.2f}  std={y_train.std():.2f}")

    scaler = StandardScaler()
    Xtr = pd.DataFrame(
        scaler.fit_transform(X_train), columns=FEATURE_NAMES
    )
    Xte = pd.DataFrame(
        scaler.transform(X_test), columns=FEATURE_NAMES
    )
    print(f"Scaled. Train: {len(Xtr)}   Test: {len(Xte)}\n")
    return Xtr, Xte, y_train, y_test


# 2. gamma - from translog scm elasticities

def compute_gamma(X_train: pd.DataFrame,
                  verbose: bool = True) -> Dict[str, float]:
    """
    Compute gamma_i from translog SCM analytic elasticities.

    Elasticity_i = d ln(Yield) / d ln(X_i) evaluated at E[ln X].
    gamma_i = Elasticity_i / sum_j(Elasticity_j)

    This is completely model-agnostic and fixed throughout the experiment.
    """
    c   = TRANSLOG_COEFFS
    lnm = {f: float(np.log(X_train[f]).mean()) for f in FEATURE_NAMES}
    lnN, lnI = lnm["Nitrogen"], lnm["Irrigation"]
    lnP, lnH = lnm["Precipitation"], lnm["Herbicide"]

    elasticities = {
        "Nitrogen":
            c["betaN"]  + c["betaNN"]*lnN + c["betaNI"]*lnI
            + c["betaNP"]*lnP + c["betaNH"]*lnH,
        "Irrigation":
            c["betaI"]  + c["betaII"]*lnI + c["betaNI"]*lnN
            + c["betaIP"]*lnP + c["betaIH"]*lnH,
        "Precipitation":
            c["betaP"]  + c["betaPP"]*lnP + c["betaNP"]*lnN
            + c["betaIP"]*lnI + c["betaPH"]*lnH,
        "Herbicide":
            c["betaH"]  + c["betaHH"]*lnH + c["betaNH"]*lnN
            + c["betaIH"]*lnI + c["betaPH"]*lnP,
    }

    total = sum(elasticities.values())
    gamma = {f: e / total for f, e in elasticities.items()}

    if verbose:
        print("gamma_i from translog SCM elasticities:")
        for f in FEATURE_NAMES:
            print(f"  {f:<18}  elasticity={elasticities[f]:.6f}  "
                  f"gamma={gamma[f]:.6f}")
        print(f"  {'sum':<18}  {total:.6f}  {sum(gamma.values()):.6f}\n")

    return gamma


# 3. build known dag

def build_known_dag() -> nx.DiGraph:
    """
    Construct the feature-only DAG.
    The yield graph has NO inter-feature edges — all features are root nodes.
    This means the SEM only models the MARGINAL distributions of each feature,
    not any structural dependencies between them.
    """
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for u, v in CAUSAL_GRAPH:
        if v != "Yield":
            G.add_edge(u, v)
    assert nx.is_directed_acyclic_graph(G), "Graph has a cycle!"
    print(f"Known DAG: {G.number_of_nodes()} nodes, "
          f"{G.number_of_edges()} inter-feature edges (none — all root nodes).")
    print(f"  All features are independent root nodes in the DAG.")
    print(f"  SEM captures only marginal distributions (mu, sigma) per feature.\n")
    return G


# 4. sem - fit and perturb

class SEM:
    """
    Linear-Gaussian SEM for the yield feature DAG.

    SPECIAL CASE: All yield features are root nodes (no inter-feature edges).
    Each mechanism is therefore:
        Xi ~ N(mu_i, sigma_i^2)
    where mu_i = E[Xi] and sigma_i = Std[Xi] (on the scaled training data).

    ARM 1 PERTURBATION (marginal-parameter version):
        mu'  = mu  * (1 + epsilon * delta_mu),    delta_mu  ~ U(-1, 1)
        sig' = sig * (1 + epsilon * |delta_sig|), delta_sig ~ U(-1, 1)
        (|delta_sig| keeps sigma positive; near-zero params are protected)

    This tests sensitivity of the interventional distributions
    E[f(X) | do(X_S = x_S)] to errors in the estimated marginal statistics.
    For root nodes this is the only source of SEM-parameter uncertainty.
    """

    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = dag
        self.features = list(features)
        self.mechs:   Dict[str, dict] = {}
        self.topo:    List[str]       = []

    def fit(self, X: pd.DataFrame,
            subsample_idx: Optional[np.ndarray] = None) -> "SEM":
        Xf = X.iloc[subsample_idx] if subsample_idx is not None else X
        self.topo = list(nx.topological_sort(self.dag))

        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                # Root node: store marginal mu and sigma
                self.mechs[var] = {
                    "parents":   [],
                    "intercept": float(Xf[var].mean()),   # mu
                    "coef":      np.array([]),
                    "sigma":     float(max(Xf[var].std(ddof=0), 1e-9)),  # sigma
                }
            else:
                # Non-root node (won't occur in this DAG, but kept for safety)
                lr    = LinearRegression().fit(Xf[parents].values, Xf[var].values)
                resid = Xf[var].values - lr.predict(Xf[parents].values)
                self.mechs[var] = {
                    "parents":   parents,
                    "intercept": float(lr.intercept_),
                    "coef":      lr.coef_.astype(float).copy(),
                    "sigma":     float(max(np.std(resid, ddof=0), 1e-9)),
                }
        return self

    def perturbed_mechs(self, epsilon: float,
                        pert_rng: np.random.Generator) -> Dict[str, dict]:
        """
        Multiplicative perturbation of SEM parameters.

        For root nodes (all nodes in this DAG):
          - mu is perturbed bidirectionally: mu * (1 + eps * delta)
          - sigma is perturbed upward only: sig * (1 + eps * |delta|)
            (sigma must remain positive)
          - Both mu and sigma are protected below WEAK_PARAM_THRESHOLD.

        For non-root nodes (structural coefficients):
          - Standard multiplicative perturbation of beta_ij.
        """
        mechs = {}
        for var, m in self.mechs.items():
            mc = copy.deepcopy(m)
            if epsilon > 0.0:
                # Perturb intercept (mu for root nodes)
                if abs(mc["intercept"]) >= WEAK_PARAM_THRESHOLD:
                    delta_mu    = pert_rng.uniform(-1.0, 1.0)
                    mc["intercept"] = mc["intercept"] * (1.0 + epsilon * delta_mu)

                # Perturb sigma (always positive)
                if mc["sigma"] >= WEAK_PARAM_THRESHOLD:
                    delta_sig = pert_rng.uniform(-1.0, 1.0)
                    mc["sigma"] = mc["sigma"] * (1.0 + epsilon * abs(delta_sig))

                # Perturb structural coefficients (non-root nodes)
                if mc["coef"].size > 0:
                    new_coef = mc["coef"].copy()
                    for k, c in enumerate(mc["coef"]):
                        if abs(c) >= WEAK_PARAM_THRESHOLD:
                            delta       = pert_rng.uniform(-1.0, 1.0)
                            new_coef[k] = c * (1.0 + epsilon * delta)
                    mc["coef"] = new_coef
            mechs[var] = mc
        return mechs

    def sample_intervention(self, S: Set[str], x_obs: pd.Series,
                            mechs: Dict[str, dict],
                            noise_rng: np.random.Generator) -> Dict[str, float]:
        """Sample one interventional world do(X_S = x_S)."""
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            m   = mechs[var]
            mu  = m["intercept"]
            sig = m["sigma"]
            if m["parents"]:
                pv  = np.array([x_new.get(p, float(x_obs[p])) for p in m["parents"]])
                mu += float(pv @ m["coef"])
            x_new[var] = mu if sig < 1e-9 else float(noise_rng.normal(mu, sig))
        return x_new


# 5. causal shap value function - regression

def _predict_scalar(model, X_mat: np.ndarray) -> float:
    return float(model.predict(X_mat)[0])


def _E_f(model, X_train: pd.DataFrame) -> float:
    return float(model.predict(X_train.values).mean())


def _v_coalition(S: Set[str], x_obs: pd.Series,
                 features: List[str],
                 sems: List[SEM], mechs_list: List[Dict],
                 model, M_per_sem: int,
                 noise_seeds: List[int]) -> float:
    """
    Estimate v(S) = E[f(X) | do(X_S = x_S)] by Monte Carlo.
    noise_seeds are SHARED so Gaussian draws are identical across all runs.
    """
    total, count, seed_idx = 0.0, 0, 0
    for sem, mechs in zip(sems, mechs_list):
        for _ in range(M_per_sem):
            noise_rng = np.random.default_rng(
                noise_seeds[seed_idx % len(noise_seeds)]
            )
            seed_idx += 1
            xs    = sem.sample_intervention(S, x_obs, mechs, noise_rng)
            X_mat = np.array([[xs[f] for f in features]], dtype=float)
            total += _predict_scalar(model, X_mat)
            count += 1
    return total / count if count > 0 else 0.0


# 6. causal shap - per instance and global

def _shap_instance(x: pd.Series, features: List[str],
                   sems: List[SEM], mechs_list: List[Dict],
                   gamma: Dict[str, float], model,
                   Ef: float, n_perms: int, M_per_sem: int,
                   perm_seeds: List[int],
                   noise_seeds: List[int]) -> np.ndarray:
    """
    Causal SHAP for one instance (permutation estimator).
    perm_seeds and noise_seeds are identical across baseline and all perturbed
    runs — only mechs_list changes, isolating SEM-parameter sensitivity.
    """
    X_mat_x = np.array([[x[f] for f in features]], dtype=float)
    fx       = _predict_scalar(model, X_mat_x)
    phi      = np.zeros(len(features))

    for perm_idx in range(n_perms):
        perm_rng = np.random.default_rng(perm_seeds[perm_idx])
        perm     = list(perm_rng.permutation(features))
        S: Set[str] = set()
        v_prev = _v_coalition(S, x, features, sems, mechs_list,
                              model, M_per_sem, noise_seeds)
        for feat in perm:
            S_next = S | {feat}
            v_curr = _v_coalition(S_next, x, features, sems, mechs_list,
                                  model, M_per_sem, noise_seeds)
            phi[features.index(feat)] += gamma[feat] * (v_curr - v_prev)
            v_prev, S = v_curr, S_next

    phi /= n_perms

    s = float(np.sum(phi))
    if abs(s) > 1e-12:
        scalar = (fx - Ef) / s
        if abs(scalar) > 20:
            print(f"    [WARN] rescale scalar={scalar:.1f}  — increase n_perms")
        phi = phi * scalar

    return phi


def _shap_global(X_test: pd.DataFrame, features: List[str],
                 sems: List[SEM], mechs_list: List[Dict],
                 gamma: Dict[str, float], model, Ef: float,
                 instance_indices: np.ndarray,
                 n_perms: int, M_per_sem: int,
                 perm_seeds: List[int], noise_seeds: List[int],
                 verbose: bool = False) -> np.ndarray:
    """Global Causal SHAP — returns (k × n_features) matrix."""
    phis = []
    for t, i in enumerate(instance_indices):
        if verbose:
            print(f"    instance {t+1}/{len(instance_indices)}", flush=True)
        phis.append(_shap_instance(
            X_test.iloc[i], features, sems, mechs_list,
            gamma, model, Ef,
            n_perms, M_per_sem, perm_seeds, noise_seeds,
        ))
    return np.array(phis)


# 7. arm 1 - adversarial epsilon sweep

def _arm1_sweep(baseline_phi: np.ndarray,
                X_test: pd.DataFrame, features: List[str],
                sems: List[SEM], gamma: Dict[str, float],
                model, Ef: float,
                instance_indices: np.ndarray,
                perm_seeds: List[int], noise_seeds: List[int],
                master_rng: np.random.Generator,
                cfg: dict) -> List[dict]:
    phi0_mean = baseline_phi.mean(axis=0)
    rows      = []

    for eps in cfg["epsilons"]:
        print(f"  eps={eps:.3f} ...", end="", flush=True)
        trial_mads, trial_rhos, trial_feat_diffs = [], [], []

        for _ in range(cfg["n_trials"]):
            pert_rng        = np.random.default_rng(int(master_rng.integers(0, 2**32)))
            perturbed_mechs = [sem.perturbed_mechs(eps, pert_rng) for sem in sems]

            phi_eps      = _shap_global(
                X_test, features, sems, perturbed_mechs, gamma,
                model, Ef,
                instance_indices=instance_indices,
                n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
                perm_seeds=perm_seeds, noise_seeds=noise_seeds,
            )
            phi_eps_mean = phi_eps.mean(axis=0)
            diff         = np.abs(phi_eps_mean - phi0_mean)
            trial_mads.append(float(diff.mean()))
            trial_feat_diffs.append(diff)
            rho, _ = spearmanr(np.abs(phi0_mean), np.abs(phi_eps_mean))
            trial_rhos.append(float(rho) if not np.isnan(rho) else 1.0)

        rows.append({
            "epsilon":         eps,
            "mad_mean":        float(np.mean(trial_mads)),
            "mad_std":         float(np.std(trial_mads)),
            "per_feature_mad": np.stack(trial_feat_diffs).mean(axis=0),
            "spearman_mean":   float(np.mean(trial_rhos)),
            "spearman_std":    float(np.std(trial_rhos)),
        })
        r = rows[-1]
        print(f"  MAD={r['mad_mean']:.5f} ± {r['mad_std']:.5f}"
              f"   rho={r['spearman_mean']:.4f} ± {r['spearman_std']:.4f}")

    return rows


def _check_monotonicity(rows: List[dict]) -> None:
    noise_floor = next((r["mad_mean"] for r in rows if r["epsilon"] == 0.0), 0.0)
    nonzero     = [r for r in rows if r["epsilon"] > 0.0]
    corrected   = [max(0.0, r["mad_mean"] - noise_floor) for r in nonzero]
    eps_vals    = [r["epsilon"] for r in nonzero]
    violations  = [
        (eps_vals[i-1], eps_vals[i], corrected[i-1], corrected[i])
        for i in range(1, len(corrected))
        if corrected[i] < corrected[i-1] * 0.90
    ]
    if violations:
        print("\n  WARNING: MONOTONICITY VIOLATIONS detected:")
        for v in violations:
            print(f"    eps={v[0]:.2f}->{v[1]:.2f}: MAD {v[2]:.5f}->{v[3]:.5f}")
        print("  -> Increase n_perms / M_per_sem before trusting these results.\n")
    else:
        print("\n  OK: Monotonicity holds — results are reliable.\n")


# 8. arm 2 - estimation-noise sweep

def _arm2_sweep(baseline_phi: np.ndarray,
                X_train: pd.DataFrame, X_test: pd.DataFrame,
                features: List[str], dag: nx.DiGraph,
                gamma: Dict[str, float], model, Ef: float,
                instance_indices: np.ndarray,
                perm_seeds: List[int], noise_seeds: List[int],
                master_rng: np.random.Generator,
                cfg: dict) -> List[dict]:
    """
    For each subsample size n:
      - Draw a random subsample of X_train of size n.
      - Refit SEM marginal statistics (mu, sigma) on that subsample.
      - Gamma is NOT refit — it comes from the fixed translog SCM.
      - Compute global Causal SHAP with fixed seeds.
      - Measure MAD(phi_n, phi_full).
    """
    phi0_mean = baseline_phi.mean(axis=0)
    n_full    = len(X_train)
    rows      = []

    for n in cfg["subsample_sizes"]:
        if n >= n_full:
            print(f"  n={n}: skipping (>= full training size {n_full})")
            continue
        print(f"  n={n} ...", end="", flush=True)

        sub_idx = master_rng.choice(n_full, size=n, replace=False)
        sems_n  = [SEM(dag, features).fit(X_train, subsample_idx=sub_idx)]

        phi_n      = _shap_global(
            X_test, features, sems_n, [s.mechs for s in sems_n], gamma,
            model, Ef,
            instance_indices=instance_indices,
            n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
            perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        )
        phi_n_mean = phi_n.mean(axis=0)
        diff       = np.abs(phi_n_mean - phi0_mean)
        rho, _     = spearmanr(np.abs(phi0_mean), np.abs(phi_n_mean))

        rows.append({
            "n":               n,
            "mad_mean":        float(diff.mean()),
            "per_feature_mad": diff,
            "spearman":        float(rho) if not np.isnan(rho) else 1.0,
        })
        print(f"  MAD={diff.mean():.5f}   rho={rows[-1]['spearman']:.4f}")

    return rows


# 9. plots

def _plot_arm1(rows: List[dict], features: List[str],
               phi0_mean: np.ndarray, cfg: dict) -> Optional[float]:
    noise_floor = next((r["mad_mean"] for r in rows if r["epsilon"] == 0.0), 0.0)
    eps_vals    = [r["epsilon"]       for r in rows]
    mad_means   = [r["mad_mean"]      for r in rows]
    mad_stds    = [r["mad_std"]       for r in rows]
    corrected   = [max(0.0, m - noise_floor) for m in mad_means]
    rho_means   = [r["spearman_mean"] for r in rows]
    rho_stds    = [r["spearman_std"]  for r in rows]

    ref       = np.abs(phi0_mean).mean()
    threshold = cfg["breakdown_pct"] * ref if ref > 1e-10 else None
    eps_star  = None
    if threshold:
        for r, c in zip(rows, corrected):
            if r["epsilon"] == 0.0: continue
            if c > threshold:
                eps_star = r["epsilon"]; break

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(
        "ARM 1: SEM Parameter Robustness — Yield Regression\n"
        "(known causal graph fixed; marginal SEM parameters perturbed)\n"
        "Note: all features are root nodes — perturbation targets (μ, σ)",
        fontsize=11, fontweight="bold"
    )

    # Raw MAD
    ax = axes[0, 0]
    lo = np.array(mad_means) - np.array(mad_stds)
    hi = np.array(mad_means) + np.array(mad_stds)
    ax.plot(eps_vals, mad_means, "o-", color="#185FA5", lw=2, ms=6,
            label=f"MAD mean ({cfg['n_trials']} trials)")
    ax.fill_between(eps_vals, lo, hi, alpha=0.18, color="#185FA5", label="±1 std")
    ax.axhline(noise_floor, color="#888", ls=":", lw=1.4,
               label=f"noise floor (ε=0): {noise_floor:.5f}")
    if threshold:
        ax.axhline(threshold, color="#A32D2D", ls="--", lw=1.3,
                   label=f"5% of |φ₀|: {threshold:.5f}")
    ax.set_xlabel("ε  (relative parameter perturbation)")
    ax.set_ylabel("MAD(φ_ε − φ₀)")
    ax.set_title("Raw MAD vs ε")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    # Noise-corrected MAD
    ax = axes[0, 1]
    ax.plot(eps_vals, corrected, "s-", color="#0F6E56", lw=2, ms=6,
            label="MAD − noise floor")
    if threshold:
        ax.axhline(threshold, color="#A32D2D", ls="--", lw=1.3,
                   label=f"5% of |φ₀|: {threshold:.5f}")
    if eps_star:
        ax.axvline(eps_star, color="#A32D2D", ls=":", lw=1.5,
                   label=f"ε* = {eps_star:.2f}")
    ax.set_xlabel("ε  (relative parameter perturbation)")
    ax.set_ylabel("Noise-corrected MAD")
    ax.set_title(f"Corrected MAD vs ε   (ε* = {eps_star})")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    # Per-feature heatmap
    ax = axes[1, 0]
    mat   = np.stack([r["per_feature_mad"] for r in rows], axis=1)
    order = np.argsort(-mat.sum(axis=1))
    mat_s = mat[order]
    fs    = [features[i] for i in order]
    im    = ax.imshow(mat_s, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(np.arange(len(eps_vals)))
    ax.set_xticklabels([f"{e:.2f}" for e in eps_vals], rotation=30,
                       ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(fs))); ax.set_yticklabels(fs, fontsize=9)
    for i in range(mat_s.shape[0]):
        for j in range(mat_s.shape[1]):
            v = mat_s[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7,
                    color="white" if v > mat_s.max() * 0.6 else "black")
    plt.colorbar(im, ax=ax, label="|Δφᵢ|")
    ax.set_title("Per-feature |Δφᵢ| heatmap\n(sorted by total sensitivity)")

    # Spearman rho
    ax = axes[1, 1]
    lo = np.array(rho_means) - np.array(rho_stds)
    hi = np.array(rho_means) + np.array(rho_stds)
    ax.plot(eps_vals, rho_means, "D-", color="#0F6E56", lw=2, ms=6,
            label="Spearman ρ mean")
    ax.fill_between(eps_vals, lo, hi, alpha=0.18, color="#0F6E56", label="±1 std")
    ax.axhline(0.9, color="#854F0B", ls="--", lw=1.3, label="ρ = 0.90")
    ax.set_ylim(-0.25, 1.08)
    ax.set_xlabel("ε  (relative parameter perturbation)")
    ax.set_ylabel("Spearman ρ  (rank correlation with φ₀)")
    ax.set_title("Feature-rank stability vs ε")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig("arm1_adversarial_yield.png", dpi=150, bbox_inches="tight")
    print("  Saved: arm1_adversarial_yield.png")
    plt.close(fig)
    return eps_star


def _plot_arm2(rows: List[dict]) -> None:
    if not rows:
        print("  Arm 2: no rows to plot."); return

    n_vals   = [r["n"]        for r in rows]
    mad_vals = [r["mad_mean"] for r in rows]
    rho_vals = [r["spearman"] for r in rows]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    fig.suptitle(
        "ARM 2: Estimation Noise (SEM subsample) — Yield Regression\n"
        "(model + graph + gamma + test instances fixed; "
        "only SEM marginal stats vary with n)",
        fontsize=11, fontweight="bold"
    )

    axes[0].plot(n_vals, mad_vals, "o-", color="#185FA5", lw=2, ms=7)
    axes[0].set_xlabel("SEM training size  n")
    axes[0].set_ylabel("MAD(φ_n − φ_full)")
    axes[0].set_title("Attribution drift vs SEM data size")
    axes[0].grid(alpha=0.3); axes[0].set_xticks(n_vals)

    axes[1].plot(n_vals, rho_vals, "s-", color="#0F6E56", lw=2, ms=7)
    axes[1].axhline(0.9, color="#854F0B", ls="--", lw=1.3, label="ρ = 0.90")
    axes[1].set_ylim(-0.25, 1.08)
    axes[1].set_xlabel("SEM training size  n")
    axes[1].set_ylabel("Spearman ρ  (rank stability)")
    axes[1].set_title("Feature-rank stability vs SEM data size")
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3); axes[1].set_xticks(n_vals)

    fig.tight_layout()
    fig.savefig("arm2_estimation_yield.png", dpi=150, bbox_inches="tight")
    print("  Saved: arm2_estimation_yield.png")
    plt.close(fig)


def _plot_gamma_sensitivity(arm1_rows: List[dict],
                            gamma: Dict[str, float],
                            features: List[str]) -> None:
    """
    Scatter: gamma_i vs per-feature sensitivity at highest epsilon.

    For the yield DAG: gamma_i is derived from the translog elasticity,
    which reflects how strongly each input drives yield.
    Features with higher elasticity (Irrigation > Nitrogen > Precipitation
    > Herbicide) should be most sensitive to perturbation of their
    marginal distribution parameters.
    """
    max_row     = max((r for r in arm1_rows if r["epsilon"] > 0.0),
                      key=lambda r: r["epsilon"])
    sensitivity = max_row["per_feature_mad"]
    gamma_vals  = np.array([gamma[f] for f in features])
    corr, _     = spearmanr(gamma_vals, sensitivity)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        "Mechanistic check — Yield Regression\n"
        f"Spearman ρ (γᵢ vs sensitivity at ε=max): {corr:.3f}  "
        f"(positive = high-γ features most sensitive, as expected)",
        fontsize=11, fontweight="bold"
    )

    # Scatter
    ax = axes[0]
    ax.scatter(gamma_vals, sensitivity, s=140, color="#185FA5", zorder=3)
    for f, gv, sv in zip(features, gamma_vals, sensitivity):
        ax.annotate(f, (gv, sv), textcoords="offset points",
                    xytext=(6, 4), fontsize=10)
    if len(features) > 2:
        c  = np.polyfit(gamma_vals, sensitivity, 1)
        xs = np.linspace(gamma_vals.min(), gamma_vals.max(), 100)
        ax.plot(xs, np.polyval(c, xs), "--", color="#A32D2D", lw=1.5,
                label="OLS trend")
    ax.set_xlabel("γᵢ  (normalised translog elasticity)", fontsize=11)
    ax.set_ylabel("|Δφᵢ| at ε=max  (attribution shift)", fontsize=11)
    ax.set_title("γᵢ vs attribution sensitivity")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # Bar chart comparison
    ax = axes[1]
    order  = np.argsort(gamma_vals)[::-1]
    feat_s = [features[i] for i in order]
    gam_s  = gamma_vals[order]
    sens_s = sensitivity[order]
    x      = np.arange(len(feat_s))
    width  = 0.38
    ax.bar(x - width/2, gam_s,  width, color="#185FA5", alpha=0.8,
           label="γᵢ (causal importance)")
    ax.bar(x + width/2, sens_s / (sens_s.max() + 1e-12), width,
           color="#E08020", alpha=0.8, label="|Δφᵢ| normalised")
    ax.set_xticks(x)
    ax.set_xticklabels(feat_s, rotation=20, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("γᵢ vs sensitivity (normalised)")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig("gamma_vs_sensitivity_yield.png", dpi=150, bbox_inches="tight")
    print("  Saved: gamma_vs_sensitivity_yield.png")
    plt.close(fig)


# 10. main pipeline

def run(cfg: dict = CFG) -> dict:
    print(f"\n{'='*72}")
    print("  CAUSAL SHAP ROBUSTNESS — YIELD REGRESSION")
    print(f"{'='*72}\n")

    master_rng = np.random.default_rng(cfg["random_seed"])

    # ── Data ─────────────────────────────────────────────────────────────
    Xtr, Xte, y_train, y_test = load_yield_data(cfg)
    features = FEATURE_NAMES

    # elasticities are defined in the original (log) space of the inputs.
    # We reload unscaled X_train here for that purpose only.
    data_raw   = np.load(cfg["data_path"], allow_pickle=True)
    X_train_raw = pd.DataFrame(data_raw["X_train"], columns=FEATURE_NAMES)
    gamma      = compute_gamma(X_train_raw, verbose=True)

    # ── Train model (frozen for entire experiment) ────────────────────────
    print("Training regression model ...")
    if HAVE_XGB:
        model = XGBRegressor(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=cfg["random_seed"], verbosity=0,
        )
        print("Using XGBRegressor")
    else:
        model = RandomForestRegressor(
            n_estimators=160, max_depth=6, random_state=cfg["random_seed"])
        print("Using RandomForestRegressor")

    model.fit(Xtr.values, y_train)
    y_pred = model.predict(Xte.values)
    print(f"RMSE={np.sqrt(mean_squared_error(y_test, y_pred)):.4f}  "
          f"MAE={mean_absolute_error(y_test, y_pred):.4f}  "
          f"R²={r2_score(y_test, y_pred):.4f}\n")

    # ── Build DAG and baseline SEM ────────────────────────────────────────
    dag       = build_known_dag()
    sems_full = [SEM(dag, features).fit(Xtr)]
    print("Baseline SEM fitted.")

    # Print SEM parameters for transparency
    print("\nBaseline SEM marginal parameters (scaled space):")
    for var in features:
        m = sems_full[0].mechs[var]
        print(f"  {var:<18}  μ = {m['intercept']:+.6f}   σ = {m['sigma']:.6f}")
    print()

    Ef = _E_f(model, Xtr)
    print(f"E[f(X)] = {Ef:.6f}")

    # ── Lock shared seeds ONCE ────────────────────────────────────────────
    k                = min(cfg["k_instances"], len(Xte))
    instance_indices = master_rng.choice(len(Xte), size=k, replace=False)
    perm_seeds       = [int(master_rng.integers(0, 2**32))
                        for _ in range(cfg["n_perms"])]
    noise_seeds      = [int(master_rng.integers(0, 2**32))
                        for _ in range(len(sems_full) * cfg["M_per_sem"] * 30)]

    print(f"\nShared seeds locked:")
    print(f"  {k} instances | {cfg['n_perms']} perm seeds | "
          f"{len(noise_seeds)} noise seeds")
    print("  (IDENTICAL for baseline + all epsilon + all n runs)\n")

    # ── Baseline SHAP ─────────────────────────────────────────────────────
    print(f"Computing baseline SHAP "
          f"(n_perms={cfg['n_perms']}, M_per_sem={cfg['M_per_sem']}, k={k}) ...")
    base_mechs   = [sem.mechs for sem in sems_full]
    baseline_phi = _shap_global(
        Xte, features, sems_full, base_mechs, gamma,
        model, Ef,
        instance_indices=instance_indices,
        n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        verbose=True,
    )
    phi0_mean = baseline_phi.mean(axis=0)
    print("\nBaseline φ₀ (sorted by |φ|):")
    for f, v in sorted(zip(features, phi0_mean), key=lambda x: -abs(x[1])):
        print(f"  {f:<20}  {v:+.6f}")

    # ── ARM 1 ─────────────────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print("ARM 1: Adversarial epsilon sweep")
    print(f"  ε = {cfg['epsilons']}")
    print(f"  {cfg['n_trials']} trials per ε")
    print("  Graph + gamma FIXED | test instances FIXED")
    print("  Only SEM marginal parameters (μ, σ) are perturbed")
    print(f"{'─'*60}")
    arm1_rows = _arm1_sweep(
        baseline_phi=baseline_phi,
        X_test=Xte, features=features,
        sems=sems_full, gamma=gamma,
        model=model, Ef=Ef,
        instance_indices=instance_indices,
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        master_rng=master_rng, cfg=cfg,
    )

    noise_floor = next((r["mad_mean"] for r in arm1_rows if r["epsilon"] == 0.0), 0.0)
    ref = np.abs(phi0_mean).mean()
    print(f"\n  MC noise floor (MAD at ε=0) : {noise_floor:.6f}")
    if ref > 1e-10:
        pct = 100 * noise_floor / ref
        print(f"  Noise floor as % of |φ₀|    : {pct:.2f}%")
        if pct > 5.0:
            print("  WARNING: noise floor > 5% — increase n_perms / M_per_sem.")
        else:
            print("  OK: noise floor is negligible.")
    _check_monotonicity(arm1_rows)

    eps_star = _plot_arm1(arm1_rows, features, phi0_mean, cfg)
    print(f"  ε* = {eps_star}  "
          f"({'breakdown reached' if eps_star else 'stable across all ε'})")

    # Save ARM 1 CSV
    records = []
    for r in arm1_rows:
        rec = {
            "epsilon":       r["epsilon"],
            "mad_mean":      r["mad_mean"],
            "mad_std":       r["mad_std"],
            "mad_corrected": max(0.0, r["mad_mean"] - noise_floor),
            "spearman_mean": r["spearman_mean"],
            "spearman_std":  r["spearman_std"],
        }
        for i, f in enumerate(features):
            rec[f"mad_{f}"] = r["per_feature_mad"][i]
        records.append(rec)
    arm1_df = pd.DataFrame(records).set_index("epsilon")
    arm1_df.to_csv("arm1_scores_yield.csv")
    print("  Saved: arm1_scores_yield.csv")
    print(arm1_df[["mad_mean", "mad_corrected", "spearman_mean"]].round(6).to_string())

    # ── ARM 2 ─────────────────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print("ARM 2: Estimation-noise sweep")
    print(f"  n = {cfg['subsample_sizes']}")
    print("  Model + graph + gamma + test instances FIXED")
    print("  Only SEM marginal statistics vary with training size n")
    print(f"{'─'*60}")
    arm2_rows = _arm2_sweep(
        baseline_phi=baseline_phi,
        X_train=Xtr, X_test=Xte,
        features=features, dag=dag,
        gamma=gamma, model=model, Ef=Ef,
        instance_indices=instance_indices,
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        master_rng=master_rng, cfg=cfg,
    )
    _plot_arm2(arm2_rows)

    records2 = []
    for r in arm2_rows:
        rec = {"n": r["n"], "mad": r["mad_mean"], "spearman": r["spearman"]}
        for i, f in enumerate(features):
            rec[f"mad_{f}"] = r["per_feature_mad"][i]
        records2.append(rec)
    arm2_df = pd.DataFrame(records2).set_index("n")
    arm2_df.to_csv("arm2_scores_yield.csv")
    print("  Saved: arm2_scores_yield.csv")
    print(arm2_df[["mad", "spearman"]].round(6).to_string())

    # ── Mechanistic plot ──────────────────────────────────────────────────
    _plot_gamma_sensitivity(arm1_rows, gamma, features)

    print(f"\n{'='*72}")
    print("  ROBUSTNESS ANALYSIS COMPLETE — Yield Regression")
    print(f"{'='*72}")
    print("\n  Claim tested: SEM marginal-parameter sensitivity under FIXED known graph.")
    print("  Scope: does NOT cover graph-structure errors.")
    print("  Note: gamma is analytic (translog SCM) — not affected by SEM fitting.\n")
    idx = int(np.argmax(np.abs(phi0_mean)))
    print(f"  Dominant feature : {features[idx]}  φ₀ = {phi0_mean[idx]:+.4f}")
    print(f"  MC noise floor   : {noise_floor:.6f}")
    if eps_star:
        print(f"  ε*               : {eps_star:.2f}  "
              f"(SEM params must be within {eps_star*100:.0f}% relative error)")
    else:
        print("  ε*               : not reached — attributions stable across all ε")

    return {
        "features":    features,
        "phi0_mean":   phi0_mean,
        "gamma":       gamma,
        "noise_floor": noise_floor,
        "eps_star":    eps_star,
        "arm1_rows":   arm1_rows,
        "arm2_rows":   arm2_rows,
    }


if __name__ == "__main__":
    run(CFG)


  CAUSAL SHAP ROBUSTNESS — YIELD REGRESSION

Loaded: X_train=(800, 4)  X_test=(200, 4)
y_train: min=27188.15  max=151221.23  mean=87032.88  std=20717.03
Scaled. Train: 800   Test: 200

gamma_i from translog SCM elasticities:
  Nitrogen            elasticity=0.660202  gamma=0.311820
  Irrigation          elasticity=0.789182  gamma=0.372739
  Precipitation       elasticity=0.520288  gamma=0.245738
  Herbicide           elasticity=0.147578  gamma=0.069703
  sum                 2.117250  1.000000

Training regression model ...
Using XGBRegressor
RMSE=5351.2529  MAE=4155.1981  R²=0.9336

Known DAG: 4 nodes, 0 inter-feature edges (none — all root nodes).
  All features are independent root nodes in the DAG.
  SEM captures only marginal distributions (mu, sigma) per feature.

Baseline SEM fitted.

Baseline SEM marginal parameters (scaled space):
  Nitrogen            μ = +0.000000   σ = 1.000000
  Irrigation          μ = -0.000000   σ = 1.000000
  Precipitation       μ = -0.000000   σ = 1.00